<div style="background:#0b1f3a;color:#fff;padding:22px 26px;border-bottom:5px solid #c9a227;border-radius:6px">
<div style="color:#c9a227;font-size:13px;letter-spacing:2px;font-weight:600">ALGOPROFESSOR AI R&amp;D SOLUTIONS</div>
<div style="font-size:30px;font-weight:700;margin:6px 0">AlgoLang Notebook</div>
<div style="font-size:15px">An agent-native programming language — every unit of work is an <b style="color:#c9a227">agent</b>.</div>
</div>

**How to open**

- **Google Colab:** go to [colab.research.google.com](https://colab.research.google.com) → *File → Upload notebook* → choose this file → *Runtime → Run all*.
- **Jupyter on your computer:** run `algo notebook` from the AlgoLang folder (or `jupyter notebook AlgoLang_Notebook.ipynb`), then *Run → Run All Cells*.

Write AlgoLang in any cell that starts with `%%algo`. Agents run offline by default (free, same answer for the same input). Section 9 switches to real Claude models.

## 0. Setup — run once

The next cells install the parser and write the AlgoLang v0.2 toolchain (compiler, runtime, Python SDK and the 120 enterprise domain packs) into a folder called `algokit/`. You can open and edit them: the language is yours to extend. After editing one, re-run it and then *Runtime → Restart* and run from here again.

In [1]:
%pip install -q lark graphviz
import os
os.makedirs("algokit/algolang", exist_ok=True)
print("✓ packages ready")

Note: you may need to restart the kernel to use updated packages.
✓ packages ready


**Grammar** — `algolang/grammar.lark` (collapsed; double-click to view)

In [2]:
%%writefile algokit/grammar.lark
// ─────────────────────────────────────────────────────────────
//  AlgoLang v0.2 grammar  —  AlgoProfessor AI R&D Solutions
//  Core rule: there are no functions. Every unit of work is an
//  `agent`; a `flow` only orchestrates agents.
// ─────────────────────────────────────────────────────────────

start: item*

?item: use_decl
     | model_decl
     | tool_decl
     | policy_decl
     | agent_decl
     | flow_decl

// use "shared/agents.algo"           import another file's models, tools and agents
// use pack insurance.claims_triage    import a domain pack from the library
// use pack legal.contract_review as legal   …and call its agents as legal.Agent(…)
use_decl: "use" STRING ["as" NAME]              -> use_file
        | "use" "pack" DOTTED ["as" NAME]       -> use_pack

// policy { … } at top level: default policy for every agent in this file
policy_decl: "policy" "{" policy_rule* "}"

// model fast = "claude-haiku-5-5"
model_decl: "model" NAME "=" STRING

// tool web.search   (declares a tool agents may be granted)
tool_decl: "tool" DOTTED

// agent Name(p: type, q: type = "x") -> type { ... }
agent_decl: "agent" NAME "(" [params] ")" "->" NAME "{" agent_clause* "}"

params: param ("," param)*
param: NAME ":" NAME ["=" literal]

?agent_clause: "role" STRING            -> role
             | "goal" STRING            -> goal
             | "uses" DOTTED ("," DOTTED)* -> uses
             | "model" NAME             -> model_ref
             | "policy" "{" policy_rule* "}" -> policy

?policy_rule: "budget" "tokens" NUMBER                       -> budget
            | "retry" NUMBER                                 -> retry
            | "require" "human" "when" "confidence" "<" NUMBER -> human_gate
            | "forbid" NAME ("," NAME)*                      -> forbid
            | "timeout" NUMBER                               -> timeout
            | "cache"                                        -> cache

// flow main(topic: text) { ... }
flow_decl: "flow" NAME "(" [params] ")" "{" stmt* "}"

?stmt: target "=" expr                                   -> assign
     | "if" cond "{" stmt* "}" ["else" "{" stmt* "}"]    -> if_stmt
     | "emit" expr ("," expr)*                           -> emit
     | call                                              -> expr_stmt

target: NAME                         -> single_target
      | "[" NAME ("," NAME)* "]"     -> multi_target

?expr: call
     | par_block
     | par_for
     | NAME                          -> var
     | literal

call: (NAME | DOTTED) "(" [args] ")"
args: arg ("," arg)*
?arg: kwarg | expr
kwarg: NAME "=" expr

// [a, b] = par { Writer(x), Critic(x) }
par_block: "par" "{" call ("," call)* "}"

// results = par for t in tasks { Worker(t) }
par_for: "par" "for" NAME "in" NAME "{" call "}"

cond: expr COMPARE expr
COMPARE: "<=" | ">=" | "==" | "!=" | "<" | ">"

?literal: STRING -> string
        | NUMBER -> number
        | "true"  -> true
        | "false" -> false

DOTTED: /[a-zA-Z_][a-zA-Z0-9_]*(\.[a-zA-Z_][a-zA-Z0-9_]*)+/
NAME: /[a-zA-Z_][a-zA-Z0-9_]*/
STRING: /"([^"\\]|\\.)*"/
NUMBER: /\d+(\.\d+)?/

COMMENT: /\/\/[^\n]*/
%ignore COMMENT
%import common.WS
%ignore WS
SEMI: ";"
%ignore SEMI


Overwriting algokit/grammar.lark


**Compiler** — `algolang/algoc.py` (collapsed; double-click to view)

In [3]:
%%writefile algokit/algoc.py
#!/usr/bin/env python3
"""
algoc — the AlgoLang compiler  (v0.2 "Enterprise")
AlgoProfessor AI R&D Solutions

    algoc check  program.algo
    algoc build  program.algo [-o out.py]
    algoc run    program.algo [--arg name=value ...] [--live]

Pipeline:  source → parse (Lark) → AST → semantic checks → Python codegen
The generated Python imports `algoruntime`, which executes agents.
"""
from __future__ import annotations

import argparse
import os
import re
import subprocess
import sys
from dataclasses import dataclass, field
from pathlib import Path

from lark import Lark, Token, Tree
from lark.exceptions import UnexpectedInput

HERE = Path(__file__).resolve().parent
GRAMMAR = (HERE / "grammar.lark").read_text()
PARSER = Lark(GRAMMAR, start="start", parser="earley", propagate_positions=True)

TYPES = {"text", "score", "list", "json", "bool"}
PACK_DIRS = [HERE / "packs"] + [Path(p) for p in os.environ.get("ALGO_PACKS", "").split(os.pathsep) if p]
BUILTIN_TOOLS = {"web.search", "file.read", "calc.eval"}
DEFAULT_MODEL = "claude-sonnet-5-5"


# ───────────────────────────── AST ─────────────────────────────
@dataclass
class Param:
    name: str
    type: str
    default: object = None
    line: int = 0


@dataclass
class Agent:
    name: str
    params: list[Param]
    returns: str
    role: str = ""
    goal: str = ""
    uses: list[str] = field(default_factory=list)
    model: str | None = None
    policy: dict = field(default_factory=dict)
    line: int = 0
    origin: str | None = None        # set for agents imported with `use`


@dataclass
class Flow:
    name: str
    params: list[Param]
    body: list
    line: int = 0


@dataclass
class Program:
    models: dict = field(default_factory=dict)
    tools: set = field(default_factory=set)
    agents: dict = field(default_factory=dict)
    flows: dict = field(default_factory=dict)
    uses: list = field(default_factory=list)      # (kind, target, alias, line)
    imports: list = field(default_factory=list)   # resolved files, for reports


@dataclass
class Diag:
    code: str
    line: int
    msg: str
    file: str | None = None          # set when the error is inside an imported file

    def __str__(self):
        where = f"{self.file} line {self.line}" if self.file else f"line {self.line}"
        return f"{where}: {self.code} {self.msg}"


class CompileError(Exception):
    def __init__(self, diags):
        self.diags = diags
        super().__init__("\n".join(map(str, diags)))


def _s(tok: Token) -> str:  # unquote STRING
    return bytes(str(tok)[1:-1], "utf-8").decode("unicode_escape")


def _line(node) -> int:
    if isinstance(node, Token):
        return node.line or 0
    return getattr(node.meta, "line", 0) if hasattr(node, "meta") else 0


# ───────────────────────── parse → AST ─────────────────────────
def literal_value(t):
    if t.data == "string":
        return _s(t.children[0])
    if t.data == "number":
        v = str(t.children[0])
        return float(v) if "." in v else int(v)
    if t.data == "true":
        return True
    if t.data == "false":
        return False
    raise ValueError(t)


def build_params(node) -> list[Param]:
    if node is None:
        return []
    out = []
    for p in node.children:
        name, typ, *rest = p.children
        default = literal_value(rest[0]) if rest and rest[0] is not None else None
        out.append(Param(str(name), str(typ), default, name.line))
    return out


def parse_policy(rules) -> dict:
    pol = {}
    for r in rules:
        if r.data == "budget":
            pol["budget_tokens"] = int(str(r.children[0]))
        elif r.data == "retry":
            pol["retry"] = int(str(r.children[0]))
        elif r.data == "timeout":
            pol["timeout_s"] = float(str(r.children[0]))
        elif r.data == "human_gate":
            pol["human_below"] = float(str(r.children[0]))
        elif r.data == "forbid":
            pol.setdefault("forbid", []).extend(str(t) for t in r.children)
        elif r.data == "cache":
            pol["cache"] = True
    return pol


def merge_policy(base: dict, own: dict) -> dict:
    """Combine two policies; the stricter rule always wins."""
    out = dict(base)
    for k, v in own.items():
        if k not in out:
            out[k] = v
        elif k in ("budget_tokens", "timeout_s"):
            out[k] = min(out[k], v)
        elif k == "human_below":
            out[k] = max(out[k], v)
        elif k == "forbid":
            out[k] = sorted(set(out[k]) | set(v), key=(out[k] + list(v)).index)
        elif k == "cache":
            out[k] = out[k] or v
        else:                      # retry: the agent's own value
            out[k] = v
    return out


def to_ast(tree: Tree) -> tuple[Program, list[Diag]]:
    prog, diags = Program(), []
    default_policy = {}
    for item in tree.children:
        if item.data in ("use_file", "use_pack"):
            target, alias = item.children
            target = _s(target) if item.data == "use_file" else str(target)
            prog.uses.append((item.data, target, str(alias) if alias else None, _line(item)))
        elif item.data == "policy_decl":
            default_policy = merge_policy(default_policy, parse_policy(item.children))
        elif item.data == "model_decl":
            name, s = item.children
            prog.models[str(name)] = _s(s)
        elif item.data == "tool_decl":
            prog.tools.add(str(item.children[0]))
        elif item.data == "agent_decl":
            name, params, ret, *clauses = item.children
            a = Agent(str(name), build_params(params), str(ret), line=name.line)
            for c in clauses:
                if c.data == "role":
                    a.role = _s(c.children[0])
                elif c.data == "goal":
                    a.goal = _s(c.children[0])
                elif c.data == "uses":
                    a.uses += [str(t) for t in c.children]
                elif c.data == "model_ref":
                    a.model = str(c.children[0])
                elif c.data == "policy":
                    a.policy = merge_policy(a.policy, parse_policy(c.children))
            if a.name in prog.agents or a.name in prog.flows:
                diags.append(Diag("E107", a.line, f"'{a.name}' is defined twice"))
            prog.agents[a.name] = a
        elif item.data == "flow_decl":
            name, params, *body = item.children
            f = Flow(str(name), build_params(params), body, line=name.line)
            if f.name in prog.agents or f.name in prog.flows:
                diags.append(Diag("E107", f.line, f"'{f.name}' is defined twice"))
            prog.flows[f.name] = f
    if default_policy:                       # file-level policy applies to every agent in the file
        for a in prog.agents.values():
            a.policy = merge_policy(default_policy, a.policy)
    return prog, diags


# ───────────────────────── semantic checks ─────────────────────
class Checker:
    """Type/effect/policy checks. Fills `diags`; returns typed var tables."""

    def __init__(self, prog: Program):
        self.p = prog
        self.diags: list[Diag] = []

    def err(self, code, line, msg):
        self.diags.append(Diag(code, line, msg))

    def run(self, library=False):
        for a in self.p.agents.values():
            if a.origin is None:             # imported agents were checked in their own file
                self.check_agent(a)
        if "main" not in self.p.flows and not library:
            self.err("E111", 1, "program has no `flow main(...)` entry point")
        for f in self.p.flows.values():
            env = {}
            for prm in f.params:
                self.check_type(prm.type, prm.line)
                env[prm.name] = prm.type
            self.check_block(f.body, env)
        return self.diags

    def check_type(self, t, line):
        if t not in TYPES:
            self.err("E105", line, f"unknown type '{t}' (use: {', '.join(sorted(TYPES))})")

    def check_agent(self, a: Agent):
        self.check_type(a.returns, a.line)
        names = set()
        for prm in a.params:
            self.check_type(prm.type, prm.line)
            names.add(prm.name)
        for ref in re.findall(r"\{(\w+)\}", a.goal):
            if ref not in names:
                self.err("E109", a.line, f"agent {a.name}: goal mentions {{{ref}}} but there is no parameter '{ref}'")
        if not a.goal:
            self.err("E112", a.line, f"agent {a.name} has no `goal` — every agent must state its work")
        for t in a.uses:  # capabilities must be declared explicitly — no ambient tools
            if t not in self.p.tools:
                self.err("E104", a.line, f"agent {a.name} uses tool '{t}', which is not declared (add `tool {t}`)")
        if a.model and a.model not in self.p.models:
            self.err("E106", a.line, f"agent {a.name}: unknown model '{a.model}'")
        for attr in a.policy.get("forbid", []):
            if attr in names:
                self.err("E108", a.line, f"agent {a.name} takes forbidden attribute '{attr}' as an input")

    # statements ---------------------------------------------------
    def check_block(self, stmts, env):
        for s in stmts:
            self.check_stmt(s, env)

    def check_stmt(self, s, env):
        line = _line(s)
        if s.data == "assign":
            target, expr = s.children
            t = self.expr_type(expr, env)
            names = [str(n) for n in target.children]
            if target.data == "multi_target":
                if expr.data != "par_block":
                    self.err("E113", line, "[a, b] = ... needs a `par { ... }` block on the right")
                    for n in names:
                        env[n] = "text"
                    return
                calls = expr.children
                if len(calls) != len(names):
                    self.err("E113", line, f"{len(names)} names but par block has {len(calls)} agents")
                for n, c in zip(names, calls):
                    env[n] = self.call_type(c, env)
            else:
                env[names[0]] = t
        elif s.data == "if_stmt":
            cond, *rest = s.children
            self.check_cond(cond, env)
            # body / else are lists of stmts mixed; lark gives flat children with None for missing else
            for child in rest:
                if child is None:
                    continue
                if isinstance(child, Tree):
                    self.check_stmt(child, env)
        elif s.data == "emit":
            for e in s.children:
                self.expr_type(e, env)
        elif s.data == "expr_stmt":
            self.expr_type(s.children[0], env)

    def check_cond(self, cond, env):
        left, op, right = cond.children
        for side in (left, right):
            t = self.expr_type(side, env)
            if t not in ("score", "number", "bool", "any"):
                name = side.children[0] if side.data == "var" else "expression"
                self.err("E110", _line(cond), f"'{name}' is {t}, not a score — only scores/numbers can be compared")

    def expr_type(self, e, env):
        if isinstance(e, Token):
            return "any"
        if e.data == "call":
            return self.call_type(e, env)
        if e.data == "var":
            n = str(e.children[0])
            if n in ("true", "false"):      # boolean literals can parse as names
                return "bool"
            if n not in env:
                self.err("E103", _line(e.children[0]), f"undefined variable '{n}'")
                return "any"
            return env[n]
        if e.data == "par_block":
            for c in e.children:
                self.call_type(c, env)
            return "list"
        if e.data == "par_for":
            var, src, call = e.children
            if str(src) not in env:
                self.err("E103", src.line, f"undefined variable '{src}'")
            elif env[str(src)] not in ("list", "any"):
                self.err("E114", src.line, f"`par for` needs a list, but '{src}' is {env[str(src)]}")
            inner = dict(env)
            inner[str(var)] = "text"
            self.call_type(call, inner)
            return "list"
        if e.data in ("string",):
            return "text"
        if e.data == "number":
            return "number"
        if e.data in ("true", "false"):
            return "bool"
        if e.data == "kwarg":
            return self.expr_type(e.children[1], env)
        return "any"

    def call_type(self, c, env):
        name_tok, args = c.children
        name = str(name_tok)
        line = name_tok.line
        arg_nodes = args.children if args is not None else []
        if name not in self.p.agents:
            hint = "it is a flow; flows orchestrate and cannot be called" if name in self.p.flows else \
                   "declare it with `agent " + name + "(...) -> type { goal \"...\" }`"
            self.err("E101", line, f"'{name}' is not an agent — in AlgoLang every unit of work is an agent ({hint})")
            for a in arg_nodes:
                self.expr_type(a, env)
            return "any"
        ag = self.p.agents[name]
        pnames = [p.name for p in ag.params]
        pos = [a for a in arg_nodes if not (isinstance(a, Tree) and a.data == "kwarg")]
        kws = [a for a in arg_nodes if isinstance(a, Tree) and a.data == "kwarg"]
        if len(pos) > len(pnames):
            self.err("E102", line, f"{name} takes {len(pnames)} argument(s) ({', '.join(pnames)}), got {len(pos)} positional")
        bound = set(pnames[: len(pos)])
        for k in kws:
            kn = str(k.children[0])
            if kn not in pnames:
                self.err("E102", line, f"{name} has no parameter '{kn}'")
            elif kn in bound:
                self.err("E102", line, f"{name}: parameter '{kn}' given twice")
            bound.add(kn)
        for p in ag.params:
            if p.name not in bound and p.default is None:
                self.err("E102", line, f"{name}: missing required argument '{p.name}'")
        for a in arg_nodes:
            self.expr_type(a, env)
        return ag.returns


# ───────────────────────── code generation ─────────────────────
def pyname(name: str) -> str:
    """Agent names may be dotted after `use … as x`; make them valid Python names."""
    return name.replace(".", "__")

class CodeGen:
    def __init__(self, prog: Program, src_name: str):
        self.p, self.src, self.out = prog, src_name, []

    def w(self, line="", ind=0):
        self.out.append("    " * ind + line)

    def gen(self) -> str:
        p = self.p
        self.w(f"# Generated by algoc v0.2 from {self.src} — do not edit; edit the .algo source.")
        self.w("import os, sys")
        self.w(f"sys.path.insert(0, os.environ.get('ALGO_HOME') or {str(HERE)!r})")
        self.w("from algoruntime import Runtime, AgentSpec")
        self.w()
        self.w(f"rt = Runtime(program={Path(self.src).stem!r}, **globals().get('__algo_runtime__', {{}}))")
        for n, m in p.models.items():
            self.w(f"rt.model({n!r}, {m!r})")
        for t in sorted(p.tools):
            self.w(f"rt.tool({t!r})")
        self.w()
        for a in p.agents.values():
            params = [(x.name, x.type, x.default) for x in a.params]
            self.w(f"{pyname(a.name)} = rt.agent(AgentSpec(")
            self.w(f"name={a.name!r}, params={params!r}, returns={a.returns!r},", 1)
            self.w(f"role={a.role!r},", 1)
            self.w(f"goal={a.goal!r},", 1)
            self.w(f"uses={a.uses!r}, model={a.model!r}, policy={a.policy!r}))", 1)
            self.w()
        for f in p.flows.values():
            args = ", ".join(x.name for x in f.params)
            self.w(f"def flow_{f.name}({args}):")
            self.w(f"rt.begin_flow({f.name!r}, {{{', '.join(f'{x.name!r}: {x.name}' for x in f.params)}}})", 1)
            self.block(f.body, 1)
            self.w("return rt.end_flow()", 1)
            self.w()
        main = p.flows.get("main")
        if main:
            self.w('if __name__ == "__main__":')
            self.w(f"rt.cli(flow_main, {[x.name for x in main.params]!r})", 1)
        return "\n".join(self.out) + "\n"

    def block(self, stmts, ind):
        if not stmts:
            self.w("pass", ind)
        for s in stmts:
            self.stmt(s, ind)

    def stmt(self, s, ind):
        if s.data == "assign":
            target, expr = s.children
            names = [str(n) for n in target.children]
            lhs = ", ".join(names) if len(names) > 1 else names[0]
            self.w(f"{lhs} = {self.expr(expr)}", ind)
        elif s.data == "if_stmt":
            cond, *rest = s.children
            # Lark flattens: body stmts then optional else stmts; split using meta lines is fragile,
            # so we re-walk: the grammar puts stmts of 'if' then stmts of 'else'. We recover the split
            # from the original tree via helper attached during parse.
            then_b, else_b = s._split
            self.w(f"if {self.cond(cond)}:", ind)
            self.block(then_b, ind + 1)
            if else_b:
                self.w("else:", ind)
                self.block(else_b, ind + 1)
        elif s.data == "emit":
            self.w(f"rt.emit({', '.join(self.expr(e) for e in s.children)})", ind)
        elif s.data == "expr_stmt":
            self.w(self.expr(s.children[0]), ind)

    def cond(self, c):
        l, op, r = c.children
        return f"rt.value({self.expr(l)}) {op} rt.value({self.expr(r)})"

    def expr(self, e):
        if e.data == "call":
            name, args = e.children
            ag = self.p.agents[str(name)]
            arg_nodes = args.children if args is not None else []
            parts, i = [], 0
            for a in arg_nodes:
                if isinstance(a, Tree) and a.data == "kwarg":
                    parts.append(f"{a.children[0]}={self.expr(a.children[1])}")
                else:
                    parts.append(f"{ag.params[i].name}={self.expr(a)}")
                    i += 1
            return f"{pyname(str(name))}({', '.join(parts)})"
        if e.data == "var":
            n = str(e.children[0])
            return {"true": "True", "false": "False"}.get(n, n)
        if e.data == "par_block":
            return "rt.par(" + ", ".join(f"lambda: {self.expr(c)}" for c in e.children) + ")"
        if e.data == "par_for":
            var, src, call = e.children
            return f"rt.par_map(lambda {var}: {self.expr(call)}, {src})"
        if e.data == "string":
            return repr(_s(e.children[0]))
        if e.data == "number":
            return str(e.children[0])
        if e.data == "true":
            return "True"
        if e.data == "false":
            return "False"
        raise ValueError(e.data)


def split_ifs(tree: Tree, src: str):
    """Attach (then, else) statement lists to each if_stmt.

    The grammar `"if" cond "{" stmt* "}" ["else" "{" stmt* "}"]` is flattened by Lark,
    so we locate the `else` keyword in the source text by position."""
    lines = src.splitlines(keepends=True)
    offsets = [0]
    for ln in lines:
        offsets.append(offsets[-1] + len(ln))

    for node in tree.iter_subtrees():
        if node.data != "if_stmt":
            continue
        cond, *rest = node.children
        stmts = [c for c in rest if isinstance(c, Tree)]
        has_else = rest and rest[-1] is None  # placeholder for missing else? (lark maybe_placeholders)
        # find "else" between end of cond and end of node
        seg = src[node.meta.start_pos: node.meta.end_pos]
        # brace matching to find end of then-block
        depth, i, then_end = 0, seg.index("{", cond.meta.end_pos - node.meta.start_pos), None
        while i < len(seg):
            if seg[i] == "{":
                depth += 1
            elif seg[i] == "}":
                depth -= 1
                if depth == 0:
                    then_end = node.meta.start_pos + i
                    break
            i += 1
        then_b = [s for s in stmts if s.meta.start_pos < then_end]
        else_b = [s for s in stmts if s.meta.start_pos > then_end]
        node._split = (then_b, else_b)


# ───────────────────────── flow graph (for diagrams) ───────────
def flow_graph(prog: Program, flow: str = "main"):
    """Return (nodes, edges) describing data flow between agents in a flow.

    nodes: list of (id, label, kind, sub) with kind in
           input | agent | par | cond | branch | emit
    edges: list of (from_id, to_id, dashed)"""
    f = prog.flows["main"]
    nodes, edges, src = [], [], {}
    count = [0]

    def new(label, kind, sub=""):
        count[0] += 1
        nid = f"n{count[0]}"
        nodes.append((nid, label, kind, sub))
        return nid

    def vars_in(tree):
        return [str(t.children[0]) for t in tree.iter_subtrees() if t.data == "var"]

    def link(names, nid, exclude=(), dashed=False):
        for v in names:
            if v in exclude:
                continue
            for s in src.get(v, []):
                if (s, nid, dashed) not in edges:
                    edges.append((s, nid, dashed))

    def text(e):
        if e.data == "var":
            return str(e.children[0])
        if e.data == "number":
            return str(e.children[0])
        if e.data == "call":
            return f"{e.children[0]}(…)"
        return e.data

    for p in f.params:
        src[p.name] = [new(p.name, "input", "flow input")]

    def walk(stmts, kind, cond_id=None, merge=False):
        def bind(n, ids):
            src[n] = (src.get(n, []) + ids) if merge else ids

        for s in stmts:
            if s.data == "assign":
                target, expr = s.children
                names = [str(n) for n in target.children]
                if expr.data == "call":
                    nid = new(str(expr.children[0]), kind, f"→ {', '.join(names)}")
                    link(vars_in(expr), nid)
                    if cond_id:
                        edges.append((cond_id, nid, True))
                    for n in names:
                        bind(n, [nid])
                elif expr.data == "par_block":
                    ids = []
                    for i, c in enumerate(expr.children):
                        sub = f"→ {names[i]}" if target.data == "multi_target" and i < len(names) else "parallel"
                        nid = new(str(c.children[0]), "par" if kind == "agent" else kind, sub + " · par")
                        link(vars_in(c), nid)
                        if cond_id:
                            edges.append((cond_id, nid, True))
                        ids.append(nid)
                    if target.data == "multi_target":
                        for n, i in zip(names, ids):
                            bind(n, [i])
                    else:
                        bind(names[0], ids)
                elif expr.data == "par_for":
                    var, lst, call = expr.children
                    nid = new(f"{call.children[0]} × each", "par", f"par for {var} in {lst} → {names[0]}")
                    link([str(lst)] + vars_in(call), nid, exclude=(str(var),))
                    bind(names[0], [nid])
                elif expr.data == "var":
                    bind(names[0], list(src.get(str(expr.children[0]), [])))
            elif s.data == "if_stmt":
                cond = s.children[0]
                l, op, r = cond.children
                cid = new(f"if {text(l)} {op} {text(r)}", "cond", "decision")
                for c in cond.iter_subtrees():
                    if c.data == "call":
                        nid = new(str(c.children[0]), kind, "in condition")
                        link(vars_in(c), nid)
                        edges.append((nid, cid, False))
                link([v for v in vars_in(cond)], cid)
                then_b, else_b = s._split
                walk(then_b, "branch", cid, merge=True)
                walk(else_b, "branch", cid, merge=True)
            elif s.data == "emit":
                eid = new("emit", "emit", "flow output")
                for e in s.children:
                    if e.data == "call":                     # emit Agent(x): draw the agent too
                        nid = new(str(e.children[0]), kind, "→ emit")
                        link(vars_in(e), nid)
                        if cond_id:
                            edges.append((cond_id, nid, True))
                        edges.append((nid, eid, False))
                    else:
                        link(vars_in(e), eid)
            elif s.data == "expr_stmt":
                c = s.children[0]
                nid = new(str(c.children[0]), kind)
                link(vars_in(c), nid)

    walk(f.body, "agent")
    return nodes, edges


# ───────────────────────── driver ──────────────────────────────
def resolve_use(kind, target, base_dir: Path):
    """Find the file behind `use "x.algo"` or `use pack sector.name`."""
    if kind == "use_file":
        p = (base_dir / target)
        return p.resolve() if p.exists() else None
    rel = Path(*target.split(".")).with_suffix(".algo")
    for d in PACK_DIRS:
        if (d / rel).exists():
            return (d / rel).resolve()
    return None


def _parse(src, name, file_label):
    try:
        tree = PARSER.parse(src)
    except UnexpectedInput as e:
        ctx = e.get_context(src).rstrip()
        raise CompileError([Diag("E100", e.line, f"syntax error near column {e.column}\n{ctx}", file_label)])
    split_ifs(tree, src)
    return tree


def load_program(src, name, base_dir: Path, stack=(), file_label=None, library=False):
    """Parse a file and everything it `use`s. Returns (Program, diagnostics)."""
    tree = _parse(src, name, file_label)
    prog, diags = to_ast(tree)
    for d in diags:
        d.file = file_label
    for kind, target, alias, line in prog.uses:
        path = resolve_use(kind, target, base_dir)
        shown = f'"{target}"' if kind == "use_file" else f"pack {target}"
        if path is None:
            hint = "check the path" if kind == "use_file" else "see `algo domains` for available packs"
            diags.append(Diag("E115", line, f"cannot find {shown} ({hint})", file_label))
            continue
        if str(path) in stack:
            diags.append(Diag("E116", line, f"circular use of {shown}", file_label))
            continue
        label = target if kind == "use_file" else f"pack {target}"
        try:
            sub, sub_diags = load_program(path.read_text(encoding="utf-8"), path.name, path.parent,
                                          stack + (str(path),), label, library=True)
        except CompileError as e:
            diags += e.diags
            continue
        diags += sub_diags
        prog.imports.append(str(path))
        prog.imports += [i for i in sub.imports if i not in prog.imports]
        for m, mid in sub.models.items():
            if m in prog.models and prog.models[m] != mid:
                diags.append(Diag("E107", line, f"model '{m}' is defined differently in {shown}", file_label))
            prog.models.setdefault(m, mid)
        prog.tools |= sub.tools
        for a in sub.agents.values():
            new_name = f"{alias}.{a.name}" if alias else a.name
            if new_name in prog.agents:
                diags.append(Diag("E107", line, f"agent '{new_name}' from {shown} is already defined — "
                                                f"import it with `as <name>` to keep both", file_label))
                continue
            a.name, a.origin = new_name, a.origin or label
            prog.agents[new_name] = a
    # local definitions come after imports, so re-check local names against imported ones
    checker = Checker(prog)
    for d in checker.run(library=library):
        d.file = d.file or file_label
        diags.append(d)
    return prog, diags


def compile_source(src: str, name: str, base_dir: Path | str | None = None):
    base = Path(base_dir) if base_dir else Path.cwd()
    me = (base / name).resolve()
    prog, diags = load_program(src, name, base, stack=(str(me),))
    if diags:
        raise CompileError(sorted(diags, key=lambda d: (d.file is not None, d.file or "", d.line)))
    return CodeGen(prog, name).gen(), prog


VERSION = "0.2.0"

TEMPLATE = '''// {name}.algo — created by `algo new`
// Run it:  algo run {name}.algo --arg "topic=Your topic here"

model smart = "claude-sonnet-5-5"
model fast  = "claude-haiku-5-5"

agent Explainer(topic: text) -> text {{
    role  "Friendly professor"
    goal  "Explain {{topic}} to a first-year student in 5 sentences"
    model fast
}}

agent QuizMaker(lesson: text) -> list {{
    role  "Exam setter"
    goal  "Write three short quiz questions on this lesson: {{lesson}}"
    model fast
}}

agent Checker(lesson: text) -> score {{
    role  "Senior reviewer"
    goal  "Score this lesson 0 to 1 for accuracy: {{lesson}}"
    model smart
    policy {{
        require human when confidence < 0.70
    }}
}}

flow main(topic: text) {{
    lesson = Explainer(topic)
    [quiz, quality] = par {{ QuizMaker(lesson), Checker(lesson) }}
    if quality < 0.75 {{
        lesson = Explainer(topic)
    }}
    emit lesson, quiz
}}
'''


def _utf8_console():
    for s in (sys.stdout, sys.stderr):
        try:
            s.reconfigure(encoding="utf-8", errors="replace")
        except Exception:  # noqa
            pass


def doctor():
    ok = True
    def line(good, label, detail):
        nonlocal ok
        ok &= good if good is not None else True
        mark = "✓" if good else ("•" if good is None else "✗")
        print(f"  {mark} {label:<22} {detail}")
    print(f"AlgoLang {VERSION} environment check")
    v = sys.version_info
    line(v >= (3, 9), "Python", f"{v.major}.{v.minor}.{v.micro} ({sys.executable})" + ("" if v >= (3, 9) else " — need 3.9+"))
    try:
        import lark
        line(True, "lark (parser)", lark.__version__)
    except ImportError:
        line(False, "lark (parser)", "missing — run: pip install lark")
    try:
        import anthropic
        line(None, "anthropic (live mode)", anthropic.__version__)
    except ImportError:
        line(None, "anthropic (live mode)", "not installed — optional; pip install anthropic for --live")
    line(None, "ANTHROPIC_API_KEY", "set" if os.environ.get("ANTHROPIC_API_KEY") else "not set — optional; needed only for --live")
    line((HERE / "algoruntime.py").exists(), "runtime", str(HERE / "algoruntime.py"))
    try:
        compile_source((HERE / "examples" / "research_brief.algo").read_text(encoding="utf-8"), "selftest.algo")
        line(True, "compiler self-test", "examples/research_brief.algo compiles")
    except Exception as e:  # noqa
        line(False, "compiler self-test", str(e)[:80])
    print("\nReady." if ok else "\nFix the ✗ items above, then run `algo doctor` again.")
    return 0 if ok else 1


MENU = """
  AlgoLang {v} Enterprise — every unit of work is an agent

   COMMAND LINE                              ENTERPRISE
     1  Create a new program                  10  Browse 120 domain packs
     2  Run a program or domain pack          11  Batch-run a program over a CSV
     3  Check a program for errors            12  Human review queue
     4  Interactive shell                     13  Usage & governance report
     5  Environment check                     14  Verify audit chains
   GRAPHICAL                                  15  Create an enterprise project
     6  AlgoLang Studio (browser IDE)
     7  Jupyter notebook
   PROMPT-BASED
     8  Prompt UI in your browser — describe a problem, get a running solution
     9  Prompt mode in this terminal
    16  Ask AI to write a program         17  Wizard (no AI needed)

     0  Exit
"""


def _pick_file():
    files = sorted(Path.cwd().glob("*.algo")) + sorted((HERE / "examples").glob("*.algo"))
    if not files:
        return input("  Path to the .algo file: ").strip()
    for n, f in enumerate(files, 1):
        where = "" if f.parent == Path.cwd() else "  (example)"
        print(f"   {n}. {f.name}{where}")
    c = input("  Choose a number or type a path: ").strip()
    if c.isdigit() and 1 <= int(c) <= len(files):
        return str(files[int(c) - 1])
    return c


def menu():
    """Numbered menu shown when `algo` is typed on its own."""
    while True:
        print(MENU.format(v=VERSION))
        try:
            c = input("  Choose 0-17: ").strip()
        except (EOFError, KeyboardInterrupt):
            print()
            return 0
        print()
        if c == "0" or c.lower() in ("q", "exit", "quit"):
            return 0
        try:
            if c == "1":
                name = input("  Name for the new program [mylesson]: ").strip() or "mylesson"
                main(["new", name])
            elif c == "2":
                f = _pick_file()
                if f:
                    main(["run", f])
            elif c == "3":
                f = _pick_file()
                if f:
                    main(["check", f])
            elif c == "4":
                main(["shell"])
            elif c == "5":
                main(["doctor"])
            elif c == "6":
                main(["studio"])
            elif c == "7":
                main(["notebook"])
            elif c == "8":
                main(["studio", "--prompt"])
            elif c == "9":
                main(["chat"])
            elif c == "10":
                q = input("  Search (e.g. 'insurance claims') or Enter to list sectors: ").strip()
                main(["domains"] + ([q] if q else []))
            elif c == "11":
                f = _pick_file()
                csvf = input("  CSV file with one row per case: ").strip()
                if f and csvf:
                    main(["batch", f, "--input", csvf])
            elif c == "12":
                main(["review"])
            elif c == "13":
                main(["report"])
            elif c == "14":
                main(["audit", "verify", "runs"])
            elif c == "15":
                d = input("  Folder for the project [my_enterprise]: ").strip() or "my_enterprise"
                main(["init", d])
            elif c == "16":
                req = input("  Describe the AI system you want: ").strip()
                if req:
                    main(["ask", req])
            elif c == "17":
                main(["wizard"])
            else:
                print("  Please type a number from 0 to 17.")
        except (EOFError, KeyboardInterrupt):
            print()
        input("\n  Press Enter to return to the menu… ")


def main(argv=None):
    _utf8_console()
    ap = argparse.ArgumentParser(prog="algo", description=f"AlgoLang {VERSION} — every unit of work is an agent")
    ap.add_argument("--version", action="version", version=f"AlgoLang {VERSION}")
    sub = ap.add_subparsers(dest="cmd", metavar="command")
    for c, h in (("check", "check a program for errors"), ("build", "compile to Python"),
                 ("run", "compile and run flow main")):
        s = sub.add_parser(c, help=h)
        s.add_argument("file")
        if c == "build":
            s.add_argument("-o", "--out")
        if c == "run":
            s.add_argument("--arg", action="append", default=[], help='flow input, e.g. --arg "topic=AI ethics"')
            s.add_argument("--live", action="store_true", help="call real Claude models (needs ANTHROPIC_API_KEY)")
            s.add_argument("--interactive", action="store_true", help="ask a human to approve low-confidence results")
            s.add_argument("--audit-dir", default=None, help="where audit logs go (default: runs/ or algo.config.json)")
    s = sub.add_parser("new", help="create a starter program")
    s.add_argument("name")
    sub.add_parser("doctor", help="check that the environment is ready")
    sub.add_parser("notebook", help="open the AlgoLang notebook in Jupyter (installs Jupyter if needed)")
    sub.add_parser("shell", help="interactive shell: type agents line by line")
    s = sub.add_parser("studio", help="AlgoLang Studio: graphical IDE in your browser")
    s.add_argument("--port", type=int, default=8765)
    s.add_argument("--no-browser", action="store_true")
    s.add_argument("--prompt", action="store_true", help="open the Prompt UI page")
    s = sub.add_parser("ui", help="Prompt UI in your browser (same as studio --prompt)")
    s.add_argument("--port", type=int, default=8765)
    s.add_argument("--no-browser", action="store_true")
    sub.add_parser("chat", help="prompt mode in the terminal: describe a problem in plain English")
    s = sub.add_parser("init", help="create an enterprise project (config, company tools, knowledge base)")
    s.add_argument("dir", nargs="?", default=".")
    s = sub.add_parser("domains", help="browse or search the 120 domain packs")
    s.add_argument("query", nargs="*")
    s.add_argument("--sector")
    s = sub.add_parser("pack", help="show a domain pack, or --copy it here to customise")
    s.add_argument("id")
    s.add_argument("--copy", action="store_true")
    s = sub.add_parser("batch", help="run a program over every row of a CSV, in parallel")
    s.add_argument("file", help="a .algo file or pack:<id>")
    s.add_argument("--input", required=True, help="CSV with a column per flow input")
    s.add_argument("--out")
    s.add_argument("--workers", type=int, default=8)
    s.add_argument("--live", action="store_true")
    s = sub.add_parser("review", help="human review queue: list, approve, reject")
    s.add_argument("action", nargs="?", choices=["list", "all", "approve", "reject"])
    s.add_argument("item", nargs="?")
    s.add_argument("--reviewer", default="")
    s.add_argument("--note", default="")
    s = sub.add_parser("audit", help="verify tamper-evident audit logs")
    s.add_argument("action", choices=["verify"])
    s.add_argument("target", nargs="?", default="runs")
    s = sub.add_parser("report", help="usage, cost and governance report from audit logs")
    s.add_argument("dir", nargs="?", default="runs")
    s.add_argument("--json", action="store_true")
    s = sub.add_parser("ask", help='agentic: AI writes a program from your description')
    s.add_argument("request", nargs="+", help='what the program should do, e.g. "review research papers"')
    s.add_argument("-o", "--out", help="file to write (default: from the request)")
    s.add_argument("--run", action="store_true", help="run the program after writing it")
    sub.add_parser("wizard", help="build a program by answering questions (no AI needed)")
    a = ap.parse_args(argv)

    if a.cmd is None:
        return menu()
    if a.cmd == "shell":
        import algoshell
        algoshell.main()
        return 0
    if a.cmd in ("studio", "ui"):
        import algostudio
        algostudio.serve(a.port, not a.no_browser, page="prompt" if (a.cmd == "ui" or a.prompt) else "")
        return 0
    if a.cmd in ("chat", "init", "domains", "pack", "batch", "review", "audit", "report"):
        import algocli_ext as X
        if a.cmd == "chat":
            return X.cmd_chat()
        if a.cmd == "init":
            return X.cmd_init(a.dir)
        if a.cmd == "domains":
            return X.cmd_domains(" ".join(a.query) or None, a.sector)
        if a.cmd == "pack":
            return X.cmd_pack(a.id, a.copy)
        if a.cmd == "batch":
            return X.cmd_batch(a.file, a.input, a.out, a.workers, a.live)
        if a.cmd == "review":
            return X.cmd_review(a.action if a.action in ("approve", "reject") else None, a.item,
                                a.reviewer, a.note, show_all=a.action == "all")
        if a.cmd == "audit":
            return X.cmd_audit_verify(a.target)
        if a.cmd == "report":
            return X.cmd_report(a.dir, a.json)
    if a.cmd == "ask":
        import algoagent
        name = algoagent.ask_cli(" ".join(a.request), a.out)
        if name and a.run:
            return main(["run", name])
        if name:
            print(f'  next:  algo run {name}')
        return 0 if name else 1
    if a.cmd == "wizard":
        import algoagent
        name, inp = algoagent.wizard()
        if input("Run it now? (y/n) [y]: ").strip().lower() in ("", "y", "yes"):
            value = input(f"Value for {inp}: ").strip() or "an example"
            return main(["run", name, "--arg", f"{inp}={value}"])
        print(f'  next:  algo run {name} --arg "{inp}=..."')
        return 0
    if a.cmd == "doctor":
        return doctor()
    if a.cmd == "notebook":
        nb = HERE / "AlgoLang_Notebook.ipynb"
        try:
            import notebook  # noqa: F401
        except ImportError:
            print("→ installing Jupyter Notebook (one time, takes a minute)…")
            rc = subprocess.call([sys.executable, "-m", "pip", "install", "--quiet", "notebook", "graphviz"])
            if rc != 0:
                print("✗ could not install Jupyter — check your internet connection", file=sys.stderr)
                return rc
        print(f"→ opening {nb.name} in your browser (press Ctrl+C here to stop Jupyter)")
        return subprocess.call([sys.executable, "-m", "notebook", str(nb)], cwd=str(HERE))
    if a.cmd == "new":
        p = Path(a.name if a.name.endswith(".algo") else a.name + ".algo")
        if p.exists():
            print(f"✗ {p} already exists", file=sys.stderr)
            return 1
        p.write_text(TEMPLATE.format(name=p.stem), encoding="utf-8")
        print(f"✓ created {p}\n  next:  algo run {p} --arg \"topic=Neural networks\"")
        return 0

    path = Path(a.file)
    from_pack = False
    if not path.exists():
        import algocli_ext
        found = algocli_ext.resolve_program(a.file)
        if not found:
            print(f"✗ file not found: {path} (for a domain pack use pack:<id>, see `algo domains`)", file=sys.stderr)
            return 1
        path, from_pack = found, True
    src = path.read_text(encoding="utf-8")
    try:
        code, prog = compile_source(src, path.name, path.resolve().parent)
    except CompileError as e:
        print(f"✗ {path.name}: {len(e.diags)} error(s)", file=sys.stderr)
        for d in e.diags:
            print("  " + str(d), file=sys.stderr)
        return 1

    if a.cmd == "check":
        print(f"✓ {path.name}: {len(prog.agents)} agents, {len(prog.flows)} flow(s), "
              f"{len(prog.tools)} tool(s) — no errors")
        return 0

    build_dir = (Path.cwd() if from_pack else path.resolve().parent) / "build"
    out = Path(getattr(a, "out", None) or (build_dir / (path.stem + ".py")))
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(code, encoding="utf-8")
    if a.cmd == "build":
        print(f"✓ compiled {path.name} → {out}")
        return 0

    env = dict(os.environ)
    env["ALGO_HOME"] = str(HERE)
    env["PYTHONIOENCODING"] = "utf-8"
    if a.audit_dir:
        env["ALGO_AUDIT_DIR"] = a.audit_dir
    if a.live:
        env["ALGO_LIVE"] = "1"
    if a.interactive:
        env["ALGO_INTERACTIVE"] = "1"
    for kv in a.arg:
        if "=" not in kv:
            print(f'✗ --arg needs name=value, e.g. --arg "topic=AI"; got: {kv}', file=sys.stderr)
            return 1
    cmd = [sys.executable, str(out)] + [f"--{kv}" for kv in a.arg]
    return subprocess.call(cmd, env=env)


if __name__ == "__main__":
    try:
        sys.exit(main())
    except BrokenPipeError:            # output piped into `head` etc.
        try:
            sys.stdout = open(os.devnull, "w")
        finally:
            sys.exit(0)


Overwriting algokit/algoc.py


**Runtime** — `algolang/algoruntime.py` (collapsed; double-click to view)

In [4]:
%%writefile algokit/algoruntime.py
"""
algoruntime — execution engine for compiled AlgoLang programs (v0.2 "Enterprise")
AlgoProfessor AI R&D Solutions

Every call in an AlgoLang program is an agent call. For each call the runtime:
  • renders the agent's role + goal into a prompt
  • applies the agent's policy *and* the organisation's policy (algo.config.json)
  • enforces forbidden attributes, token budgets, a run-level token cap, retries,
    timeouts and the human-in-the-loop gate
  • executes the agent's granted tools (built-in or registered by your company)
  • reuses cached results when caching is on
  • limits how many model calls run at once across the whole process
  • writes a hash-chained (tamper-evident) audit record
  • puts low-confidence results in a review queue for people to approve or reject

Backends
  mock  (default)  deterministic, offline — for teaching, tests and CI
  live  (ALGO_LIVE=1 + ANTHROPIC_API_KEY)  calls Claude via the anthropic SDK
"""
from __future__ import annotations

import argparse
import ast
import hashlib
import importlib.util
import json
import operator
import os
import re
import sqlite3
import sys
import threading
import time
import uuid
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, field
from pathlib import Path

VERSION = "0.2.0"


class PolicyViolation(Exception):
    pass


@dataclass
class AgentSpec:
    name: str
    params: list
    returns: str
    role: str = ""
    goal: str = ""
    uses: list = field(default_factory=list)
    model: str | None = None
    policy: dict = field(default_factory=dict)


@dataclass
class Result:
    """What every agent returns: a value plus its provenance."""
    value: object
    confidence: float
    agent: str
    call_id: str
    status: str = "ok"           # ok | escalated | approved

    def __str__(self):
        return str(self.value)


# ═════════════════════════ organisation configuration ═════════════════════════
DEFAULT_CONFIG = {
    "organisation": "",
    "policy": {                      # applied on top of every agent's own policy; stricter wins
        "forbid": [],                # protected attributes no agent may receive
        "max_budget_tokens": None,   # cap on any agent's token budget
        "min_human_below": None,     # every agent sends results below this confidence to people
        "allowed_tools": None,       # list → only these tools may be used by any agent
    },
    "max_concurrency": 16,           # model calls running at once, across all flows and batch rows
    "max_run_tokens": None,          # stop a single run that would use more tokens than this
    "cache": {"enabled": False, "path": None},      # None → <audit_dir>/cache.sqlite
    "audit_dir": "runs",
    "review_queue": None,                           # None → <audit_dir>/review_queue.jsonl
    "models": {},                    # central model governance: short name → model id override
    "tools": {"file.read": {"root": "."}, "kb.search": {"dir": "kb"}},
    "plugins": [],                   # Python files that register company tools
}

_CONFIG = None
_CONFIG_LOCK = threading.Lock()


def _deep_merge(a, b):
    out = dict(a)
    for k, v in (b or {}).items():
        out[k] = _deep_merge(out[k], v) if isinstance(v, dict) and isinstance(out.get(k), dict) else v
    return out


def config_path():
    p = os.environ.get("ALGO_CONFIG")
    if p:
        return Path(p)
    here = Path.cwd() / "algo.config.json"
    return here if here.exists() else None


def load_config(reload=False):
    """Organisation configuration from ALGO_CONFIG or ./algo.config.json (merged over defaults)."""
    global _CONFIG
    with _CONFIG_LOCK:
        if _CONFIG is None or reload:
            cfg = dict(DEFAULT_CONFIG)
            p = config_path()
            if p and p.exists():
                cfg = _deep_merge(cfg, json.loads(p.read_text(encoding="utf-8")))
            _CONFIG = cfg
            for plugin in cfg.get("plugins") or []:
                _load_plugin(plugin)
        return _CONFIG


def configure(**overrides):
    """Override configuration in code (used by the SDK)."""
    global _CONFIG
    cfg = load_config()
    with _CONFIG_LOCK:
        _CONFIG = _deep_merge(cfg, overrides)
        _SEMAPHORE_SIZE[0] = None
    return _CONFIG


_SEMAPHORE = [None]
_SEMAPHORE_SIZE = [None]


def _semaphore():
    n = int(load_config().get("max_concurrency") or 16)
    with _CONFIG_LOCK:
        if _SEMAPHORE_SIZE[0] != n:
            _SEMAPHORE[0] = threading.BoundedSemaphore(n)
            _SEMAPHORE_SIZE[0] = n
        return _SEMAPHORE[0]


# ═════════════════════════════════ tools ═════════════════════════════════════
@dataclass
class Tool:
    name: str
    fn: object
    description: str = ""


TOOLS: dict[str, Tool] = {}


def register_tool(name, description=""):
    """Decorator: make a Python function available to agents that `uses` this tool.

        @register_tool("crm.lookup", "Look up a customer record by name or ID")
        def crm_lookup(query: str) -> str: ...
    """
    def deco(fn):
        TOOLS[name] = Tool(name, fn, description or (fn.__doc__ or "").strip())
        return fn
    return deco


def _load_plugin(path):
    p = Path(path)
    if not p.is_absolute():
        base = config_path().parent if config_path() else Path.cwd()
        p = base / p
    spec = importlib.util.spec_from_file_location(f"algo_plugin_{p.stem}", p)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)


_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
        ast.Pow: operator.pow, ast.Mod: operator.mod, ast.USub: operator.neg, ast.UAdd: operator.pos}


def _safe_eval(node):
    if isinstance(node, ast.Expression):
        return _safe_eval(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_safe_eval(node.operand))
    raise ValueError("only arithmetic is allowed")


@register_tool("calc.eval", "Evaluate an arithmetic expression such as (1200 * 0.18) + 45")
def _calc(query: str) -> str:
    m = re.findall(r"[\d\.\s\+\-\*/\(\)%]+", query)
    expr = max(m, key=len).strip() if m else ""
    try:
        return f"{expr} = {_safe_eval(ast.parse(expr, mode='eval'))}"
    except Exception as e:  # noqa
        return f"could not evaluate '{expr}': {e}"


@register_tool("file.read", "Read a text file from the organisation's workspace (path relative to it)")
def _file_read(query: str) -> str:
    root = Path(load_config()["tools"].get("file.read", {}).get("root", ".")).resolve()
    m = re.search(r"[\w\-./\\]+\.(?:txt|md|csv|json|log|algo)", query)
    if not m:
        return "no file name found in the request"
    p = (root / m.group(0)).resolve()
    if not str(p).startswith(str(root)):
        return "refused: path is outside the permitted workspace"
    if not p.exists():
        return f"not found: {m.group(0)}"
    return p.read_text(encoding="utf-8", errors="replace")[:4000]


@register_tool("kb.search", "Search the organisation's knowledge base of .txt and .md documents")
def _kb_search(query: str) -> str:
    kb = Path(load_config()["tools"].get("kb.search", {}).get("dir", "kb"))
    docs = list(kb.rglob("*.txt")) + list(kb.rglob("*.md")) if kb.exists() else []
    if not docs:
        return f"knowledge base '{kb}' is empty"
    terms = {w for w in re.findall(r"[a-z0-9]{3,}", query.lower())}
    scored = []
    for d in docs:
        text = d.read_text(encoding="utf-8", errors="replace")
        for para in re.split(r"\n\s*\n", text):
            words = re.findall(r"[a-z0-9]{3,}", para.lower())
            hits = sum(1 for w in words if w in terms)
            if hits:
                scored.append((hits / (1 + len(words) ** 0.5), d.name, para.strip()))
    scored.sort(reverse=True)
    if not scored:
        return "no matching passages"
    return "\n".join(f"[{name}] {para[:400]}" for _, name, para in scored[:3])


def run_tool(name, query):
    t = TOOLS.get(name)
    if t is None:
        return None
    try:
        return str(t.fn(query))
    except Exception as e:  # noqa
        return f"tool error: {type(e).__name__}: {e}"


# ═══════════════════════════════ backends ════════════════════════════════════
def _h(*parts) -> int:
    return int(hashlib.sha256("|".join(map(str, parts)).encode()).hexdigest(), 16)


class MockBackend:
    """Deterministic offline backend. Granted, registered tools really run."""
    name = "mock"

    def complete(self, spec: AgentSpec, model: str, prompt: str, goal: str):
        tool_calls, notes = [], []
        for t in spec.uses:
            out = run_tool(t, goal)
            if out is not None:
                tool_calls.append({"tool": t, "chars": len(out)})
                notes.append(f"{t}: {out.splitlines()[0][:60] if out else ''}")
        seed = _h(spec.name, goal)
        conf = round(0.62 + (seed % 3300) / 10000, 2)           # 0.62 – 0.95
        short = (goal[:70] + "…") if len(goal) > 70 else goal
        if spec.returns == "score":
            val = round(0.55 + (seed // 7 % 4000) / 10000, 2)      # 0.55 – 0.95
        elif spec.returns == "list":
            n = 3 + seed % 2
            val = [f"item {i + 1} for: {short}" for i in range(n)]
            if spec.name.split(".")[-1].lower().startswith("plan"):
                pool = ["Novelty & contribution", "Methodology & rigour", "Experimental validity",
                        "Related-work coverage", "Clarity & presentation"]
                val = pool[: n]
        elif spec.returns == "bool":
            val = bool(seed % 2)
        elif spec.returns == "json":
            val = {"agent": spec.name, "summary": short}
        else:
            val = f"«{spec.name} #{seed % 10000:04d}» {short}"
            if notes:
                val += "  [" + " | ".join(notes) + "]"
        tokens = len(prompt) // 4 + 60
        return val, conf, tokens, tool_calls


class ClaudeBackend:
    name = "live"
    FORMATS = {
        "score": 'Reply as JSON: {"value": <number 0-1>, "confidence": <0-1>}',
        "list": 'Reply as JSON: {"value": [<strings>], "confidence": <0-1>}',
        "bool": 'Reply as JSON: {"value": true|false, "confidence": <0-1>}',
        "json": 'Reply as JSON: {"value": <object>, "confidence": <0-1>}',
        "text": 'Reply as JSON: {"value": "<text>", "confidence": <0-1>}',
    }

    def __init__(self, client=None):
        if client is None:
            import anthropic  # noqa: lazy import
            client = anthropic.Anthropic()
        self.client = client

    def complete(self, spec: AgentSpec, model: str, prompt: str, goal: str):
        max_tok = min(int(spec.policy.get("budget_tokens", 2000)), 8000)
        tools = [{"name": t.replace(".", "_"), "description": TOOLS[t].description or t,
                  "input_schema": {"type": "object", "properties": {"query": {"type": "string"}},
                                   "required": ["query"]}}
                 for t in spec.uses if t in TOOLS]
        by_api_name = {t.replace(".", "_"): t for t in spec.uses}
        messages = [{"role": "user", "content": prompt}]
        system = f"You are {spec.role or spec.name}. {self.FORMATS[spec.returns]}. Output JSON only."
        tokens, tool_calls = 0, []
        for _round in range(6):                       # at most 5 rounds of tool use
            kw = dict(model=model, max_tokens=max_tok, system=system, messages=messages)
            if tools:
                kw["tools"] = tools
            msg = self.client.messages.create(**kw)
            tokens += msg.usage.input_tokens + msg.usage.output_tokens
            if getattr(msg, "stop_reason", None) != "tool_use":
                break
            results = []
            for b in msg.content:
                if getattr(b, "type", "") == "tool_use":
                    name = by_api_name.get(b.name, b.name)
                    out = run_tool(name, (b.input or {}).get("query", "")) if name in spec.uses else \
                        "refused: this agent is not granted that tool"
                    tool_calls.append({"tool": name, "chars": len(out or "")})
                    results.append({"type": "tool_result", "tool_use_id": b.id, "content": out or ""})
            messages += [{"role": "assistant", "content": msg.content}, {"role": "user", "content": results}]
        raw = "".join(getattr(b, "text", "") for b in msg.content if getattr(b, "type", "") == "text")
        m = re.search(r"\{.*\}", raw, re.S)
        try:
            data = json.loads(m.group(0)) if m else {"value": raw, "confidence": 0.5}
        except json.JSONDecodeError:
            data = {"value": raw, "confidence": 0.5}
        return data.get("value"), float(data.get("confidence", 0.5)), tokens, tool_calls


# ═════════════════════════════════ cache ═════════════════════════════════════
class Cache:
    _lock = threading.Lock()

    def __init__(self, path):
        self.path = Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)
        with self._lock, sqlite3.connect(self.path) as db:
            db.execute("create table if not exists calls (key text primary key, value text, conf real, ts text)")

    @staticmethod
    def key(*parts):
        return hashlib.sha256("\x1f".join(map(str, parts)).encode()).hexdigest()

    def get(self, key):
        with self._lock, sqlite3.connect(self.path) as db:
            row = db.execute("select value, conf from calls where key=?", (key,)).fetchone()
        return (json.loads(row[0]), row[1]) if row else None

    def put(self, key, value, conf):
        with self._lock, sqlite3.connect(self.path) as db:
            db.execute("insert or replace into calls values (?,?,?,?)",
                       (key, json.dumps(value, default=str), conf, time.strftime("%Y-%m-%dT%H:%M:%S")))


# ══════════════════════════ tamper-evident audit log ═════════════════════════
_AUDIT_LOCK = threading.Lock()


def _last_hash(path: Path):
    if not path.exists() or path.stat().st_size == 0:
        return "0" * 64
    with open(path, "rb") as f:
        f.seek(0, 2)
        size = f.tell()
        f.seek(max(0, size - 65536))
        tail = f.read().decode("utf-8", errors="replace").strip().splitlines()
    try:
        return json.loads(tail[-1]).get("hash", "0" * 64)
    except Exception:  # noqa
        return "0" * 64


def _record_hash(prev, rec):
    body = json.dumps(rec, sort_keys=True, default=str)
    return hashlib.sha256((prev + body).encode()).hexdigest()


def append_audit(path: Path, rec: dict):
    """Append a record whose hash covers the previous record's hash (a hash chain)."""
    with _AUDIT_LOCK:
        path.parent.mkdir(parents=True, exist_ok=True)
        prev = _last_hash(path)
        rec = dict(rec, prev=prev)
        rec["hash"] = _record_hash(prev, {k: v for k, v in rec.items() if k != "hash"})
        with open(path, "a", encoding="utf-8") as f:
            f.write(json.dumps(rec, default=str) + "\n")
        return rec


def verify_audit(path):
    """Check every record's hash and link. Returns dict(ok, records, first_bad_line, reason)."""
    prev, n = "0" * 64, 0
    for n, line in enumerate(Path(path).read_text(encoding="utf-8").splitlines(), 1):
        if not line.strip():
            continue
        try:
            rec = json.loads(line)
        except json.JSONDecodeError:
            return {"ok": False, "records": n - 1, "first_bad_line": n, "reason": "not valid JSON"}
        if "hash" not in rec:
            return {"ok": False, "records": n - 1, "first_bad_line": n, "reason": "record has no hash (v0.1 log?)"}
        if rec.get("prev") != prev:
            return {"ok": False, "records": n - 1, "first_bad_line": n, "reason": "chain broken: a record was removed or reordered"}
        if _record_hash(prev, {k: v for k, v in rec.items() if k != "hash"}) != rec["hash"]:
            return {"ok": False, "records": n - 1, "first_bad_line": n, "reason": "record was modified"}
        prev = rec["hash"]
    return {"ok": True, "records": n, "first_bad_line": None, "reason": ""}


# ═══════════════════════════════ review queue ════════════════════════════════
def audit_dir():
    return Path(os.environ.get("ALGO_AUDIT_DIR") or load_config().get("audit_dir") or "runs")


def review_queue_path():
    q = load_config().get("review_queue")
    return Path(q) if q else audit_dir() / "review_queue.jsonl"


def queue_for_review(item: dict):
    append_audit(review_queue_path(), dict(type="item", status="pending", **item))


def list_reviews(status="pending", path=None):
    """Items in the review queue with their latest decision applied."""
    p = Path(path) if path else review_queue_path()
    if not p.exists():
        return []
    items = {}
    for line in p.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        r = json.loads(line)
        if r.get("type") == "item":
            items[r["id"]] = r
        elif r.get("type") == "decision" and r.get("item") in items:
            items[r["item"]].update(status=r["decision"], reviewer=r.get("reviewer"),
                                    note=r.get("note"), decided=r.get("ts"))
    out = list(items.values())
    return [i for i in out if status in (None, "all") or i["status"] == status]


def decide_review(item_id, decision, reviewer="", note="", path=None):
    if decision not in ("approved", "rejected"):
        raise ValueError("decision must be 'approved' or 'rejected'")
    p = Path(path) if path else review_queue_path()
    known = {i["id"] for i in list_reviews("all", p)}
    match = [i for i in known if i.startswith(item_id)]
    if len(match) != 1:
        raise KeyError(f"no single review item matches '{item_id}'")
    append_audit(p, dict(type="decision", item=match[0], decision=decision, reviewer=reviewer,
                         note=note, ts=time.strftime("%Y-%m-%dT%H:%M:%S")))
    return match[0]


# ═══════════════════════════════ runtime ═════════════════════════════════════
class Runtime:
    def __init__(self, program: str, quiet: bool | None = None, backend=None, live: bool | None = None,
                 audit_dir_override: str | None = None):
        self.program = program
        self.cfg = load_config()
        self.models = {}
        self.tools = set()
        self.quiet = (os.environ.get("ALGO_QUIET") == "1") if quiet is None else quiet
        self.live = (os.environ.get("ALGO_LIVE") == "1") if live is None else live
        if backend is not None:
            self.backend = backend
        elif self.live:
            if not os.environ.get("ANTHROPIC_API_KEY"):
                raise SystemExit("✗ --live needs ANTHROPIC_API_KEY to be set "
                                 "(Colab: add it under Secrets; terminal: see RUN_GUIDE.md, step 5)")
            try:
                self.backend = ClaudeBackend()
            except ImportError:
                raise SystemExit("✗ --live needs the anthropic package:  pip install anthropic")
        else:
            self.backend = MockBackend()
        self.emitted = []
        self.calls = []
        self.lock = threading.Lock()
        self.stats = {"calls": 0, "tokens": 0, "escalations": 0, "retries": 0, "blocked": 0, "cache_hits": 0,
                      "tool_calls": 0}
        self.run_id = uuid.uuid4().hex[:8]
        adir = Path(audit_dir_override) if audit_dir_override else audit_dir()
        adir.mkdir(parents=True, exist_ok=True)
        self.audit_path = adir / f"{program}.audit.jsonl"
        c = self.cfg.get("cache") or {}
        self.cache = Cache(c.get("path") or adir / "cache.sqlite")
        self.cache_all = bool(c.get("enabled"))
        if not self.quiet:
            for s in (sys.stdout, sys.stderr):
                try:
                    s.reconfigure(encoding="utf-8", errors="replace")
                except Exception:  # noqa
                    pass

    # declarations ---------------------------------------------------
    def model(self, name, model_id):
        self.models[name] = (self.cfg.get("models") or {}).get(name, model_id)

    def tool(self, name):
        self.tools.add(name)

    def agent(self, spec: AgentSpec):
        rt = self

        def call(**kwargs):
            return rt._invoke(spec, kwargs)

        call.__name__ = spec.name.replace(".", "__")
        call.spec = spec
        return call

    # helpers --------------------------------------------------------
    @staticmethod
    def value(x):
        return x.value if isinstance(x, Result) else x

    def _log(self, msg):
        if not self.quiet:
            with self.lock:
                print(msg, flush=True)

    def _audit(self, rec, output=None):
        rec.update(run=self.run_id, program=self.program, ts=time.strftime("%Y-%m-%dT%H:%M:%S"))
        rec = append_audit(self.audit_path, rec)
        with self.lock:
            self.calls.append(dict(rec, output=output))   # in-memory copy for notebooks and the SDK

    def effective_policy(self, spec):
        """The agent's policy combined with the organisation's; the stricter rule wins."""
        pol = dict(spec.policy)
        org = self.cfg.get("policy") or {}
        if org.get("forbid"):
            pol["forbid"] = list(dict.fromkeys(list(pol.get("forbid", [])) + list(org["forbid"])))
        if org.get("max_budget_tokens"):
            pol["budget_tokens"] = min(pol.get("budget_tokens", org["max_budget_tokens"]), org["max_budget_tokens"])
        if org.get("min_human_below") is not None:
            pol["human_below"] = max(pol.get("human_below", 0), org["min_human_below"])
        return pol

    # the heart: one agent call -------------------------------------
    def _invoke(self, spec: AgentSpec, kwargs):
        pol = self.effective_policy(spec)
        org = self.cfg.get("policy") or {}
        args = {}
        for name, _typ, default in spec.params:
            args[name] = self.value(kwargs[name]) if name in kwargs else default
        call_id = uuid.uuid4().hex[:6]
        model_id = self.models.get(spec.model, "claude-sonnet-5-5") if spec.model else \
            (self.cfg.get("models") or {}).get("default", "claude-sonnet-5-5")

        def fmt(v):
            if isinstance(v, list):
                return "; ".join(map(str, (self.value(x) for x in v)))
            return str(v)
        goal = re.sub(r"\{(\w+)\}", lambda m: fmt(args.get(m.group(1), m.group(0))), spec.goal)
        prompt = f"ROLE: {spec.role}\nGOAL: {goal}\nTOOLS: {', '.join(spec.uses) or 'none'}"

        def block(reason, msg):
            with self.lock:
                self.stats["blocked"] += 1
            self._audit(dict(call=call_id, agent=spec.name, status="blocked", reason=reason, policy=pol))
            raise PolicyViolation(f"{spec.name}: {msg}")

        # organisation: only approved tools
        allowed = org.get("allowed_tools")
        if allowed is not None:
            for t in spec.uses:
                if t not in allowed:
                    block(f"tool '{t}' not allowed by organisation policy", f"tool '{t}' is not allowed by organisation policy")
        # policy: forbidden attributes must not appear in the inputs
        for attr in pol.get("forbid", []):
            if re.search(rf"\b{re.escape(attr)}\b", prompt, re.I):
                block(f"forbidden attribute '{attr}'", f"input mentions forbidden attribute '{attr}'")
        # policy: token budget (pre-flight estimate)
        budget = pol.get("budget_tokens")
        if budget and len(prompt) // 4 > budget:
            block(f"prompt ~{len(prompt)//4} tokens exceeds budget {budget}",
                  f"prompt (~{len(prompt)//4} tok) exceeds budget {budget}")
        # organisation: run-level token cap
        cap = self.cfg.get("max_run_tokens")
        if cap and self.stats["tokens"] + len(prompt) // 4 > cap:
            block(f"run token cap {cap} reached", f"this run reached the organisation's token cap ({cap})")

        use_cache = self.cache_all or pol.get("cache")
        ckey = Cache.key(self.backend.name, model_id, spec.returns, spec.role, prompt) if use_cache else None
        hit = self.cache.get(ckey) if use_cache else None
        tool_calls, cached = [], False
        if hit is not None:
            val, conf = hit
            tokens, cached = 0, True
            with self.lock:
                self.stats["cache_hits"] += 1
        else:
            tries = 1 + int(pol.get("retry", 0))
            for attempt in range(1, tries + 1):
                try:
                    t0 = time.time()
                    with _semaphore():
                        val, conf, tokens, tool_calls = self.backend.complete(spec, model_id, prompt, goal)
                    dt = time.time() - t0
                    if pol.get("timeout_s") and dt > pol["timeout_s"]:
                        raise TimeoutError(f"{spec.name} exceeded {pol['timeout_s']}s")
                    break
                except PolicyViolation:
                    raise
                except Exception as e:  # noqa
                    with self.lock:
                        self.stats["retries"] += 1
                    if attempt == tries:
                        self._audit(dict(call=call_id, agent=spec.name, status="failed", error=str(e)[:500]))
                        raise
            if use_cache:
                self.cache.put(ckey, val, conf)
        res = Result(val, conf, spec.name, call_id)

        # policy: human-in-the-loop gate
        gate = pol.get("human_below")
        if gate is not None and conf < gate:
            res.status = self._human_review(spec, res, gate)
            if res.status == "escalated":
                queue_for_review(dict(id=uuid.uuid4().hex[:10], program=self.program, run=self.run_id, call=call_id,
                                      agent=spec.name, confidence=conf, threshold=gate,
                                      output=str(val)[:2000], ts=time.strftime("%Y-%m-%dT%H:%M:%S")))

        with self.lock:
            self.stats["calls"] += 1
            self.stats["tokens"] += tokens
            self.stats["tool_calls"] += len(tool_calls)
        shown = val if not isinstance(val, list) else f"[{len(val)} items]"
        flag = {"ok": "", "escalated": "  ⚠ ESCALATED to human (conf < %.2f)" % (gate or 0),
                "approved": "  ✔ human-approved"}[res.status]
        self._log(f"  → {spec.name:<18} conf={conf:.2f}  {str(shown)[:72]}{flag}{'  (cached)' if cached else ''}")
        self._audit(dict(call=call_id, agent=spec.name, model=model_id, backend=self.backend.name,
                         inputs_sha=hashlib.sha256(prompt.encode()).hexdigest()[:16],
                         returns=spec.returns, confidence=conf, tokens=tokens, status=res.status,
                         cached=cached, tools=spec.uses, tool_calls=tool_calls, policy=pol), output=val)
        return res

    def _human_review(self, spec, res, gate):
        with self.lock:
            self.stats["escalations"] += 1
        if os.environ.get("ALGO_INTERACTIVE") == "1" and not self.quiet:
            try:
                with self.lock:
                    ans = input(f"\n[HUMAN REVIEW] {spec.name} (conf {res.confidence:.2f}) → {res.value}\napprove? [y/N] ")
                return "approved" if ans.strip().lower() == "y" else "escalated"
            except Exception:  # noqa — no input available (headless run)
                return "escalated"
        return "escalated"   # queued for a human (review queue + audit log)

    # concurrency ----------------------------------------------------
    def par(self, *thunks):
        with ThreadPoolExecutor(max_workers=len(thunks)) as ex:
            return [f.result() for f in [ex.submit(t) for t in thunks]]

    def par_map(self, fn, items):
        items = self.value(items)
        if not isinstance(items, list):
            items = [items]
        with ThreadPoolExecutor(max_workers=max(1, min(len(items), 64))) as ex:
            return [f.result() for f in [ex.submit(fn, it) for it in items]]

    # flow lifecycle -------------------------------------------------
    def begin_flow(self, name, args):
        self._log(f"▶ flow {name}  run={self.run_id}  backend={self.backend.name}")
        for k, v in args.items():
            self._log(f"    {k} = {v!r}")

    def emit(self, *vals):
        self.emitted.extend(vals)

    def end_flow(self):
        s = self.stats
        if os.environ.get("ALGO_NOTEBOOK") == "1" or self.quiet:
            return [self.value(v) for v in self.emitted]
        self._log("■ emitted:")
        for v in self.emitted:
            v = self.value(v)
            if isinstance(v, list):
                for x in v:
                    self._log(f"    • {self.value(x)}")
            else:
                self._log(f"    • {v}")
        extra = f" · {s['cache_hits']} cached" if s["cache_hits"] else ""
        extra += f" · {s['tool_calls']} tool calls" if s["tool_calls"] else ""
        self._log(f"■ {s['calls']} agent calls · ~{s['tokens']} tokens · {s['retries']} retries · "
                  f"{s['escalations']} escalated · {s['blocked']} blocked{extra} · audit → {self.audit_path}")
        return [self.value(v) for v in self.emitted]

    def cli(self, flow, params):
        ap = argparse.ArgumentParser(prog=self.program)
        for p in params:
            ap.add_argument(f"--{p}")
        a = ap.parse_args()
        vals = {}
        for p in params:
            v = getattr(a, p)
            if v is None:
                if not sys.stdin.isatty():
                    self._log(f'✗ missing input: add --arg "{p}=..."')
                    sys.exit(1)
                v = input(f"{p}: ")
            vals[p] = v
        try:
            flow(**vals)
        except PolicyViolation as e:
            self._log(f"✗ POLICY VIOLATION: {e}")
            sys.exit(2)
        except Exception as e:  # noqa — model/API failures after retries
            self._log(f"✗ agent call failed: {type(e).__name__}: {str(e)[:300]}")
            self._log(f"  details are in the audit log: {self.audit_path}")
            sys.exit(3)


Overwriting algokit/algoruntime.py


**Notebook extension** — `algolang/algomagic.py` (collapsed; double-click to view)

In [5]:
%%writefile algokit/algomagic.py
"""
algomagic — run AlgoLang inside Jupyter Notebook, JupyterLab and Google Colab.
AlgoProfessor AI R&D Solutions · AlgoLang v0.1

    %load_ext algomagic

    %%algo --arg "topic=Neural networks"      run a whole program in one cell
    %%algo --define                           add agents/models/tools to the session
    %%algo --check                            only check for errors
    %%algo --graph                            also draw the flow as a diagram
    %%algo --python                           also show the generated Python
    %%algo --live                             call real Claude models
    %%algo --interactive                      ask a human to approve low-confidence results

    %algo_run examples/research_brief.algo --arg "topic=..."   run a .algo file
    %algo_list     what is defined in this session
    %algo_reset    forget all --define cells
    %algo_help     this help

After a run, these Python variables are set:
    algo_result   list of values the flow emitted
    algo_calls    list of dicts, one per agent call (agent, model, confidence, status, tokens, output…)
"""
from __future__ import annotations

import argparse
import html
import os
import re
import shlex
import shutil
import sys
from pathlib import Path

HERE = Path(__file__).resolve().parent
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))
os.environ.setdefault("ALGO_HOME", str(HERE))

from IPython.core.magic import Magics, cell_magic, line_magic, magics_class  # noqa: E402
from IPython.display import HTML, Markdown, display  # noqa: E402

import algoruntime  # noqa: E402
from algoc import PARSER, VERSION, CompileError, compile_source, flow_graph  # noqa: E402

CSS = """<style>
.algo-box{font-family:'Segoe UI',system-ui,-apple-system,sans-serif;border:1px solid #0b1f3a;
  border-left:6px solid #c9a227;border-radius:6px;background:#ffffff;color:#1b1b1b;margin:8px 0;
  overflow:hidden;max-width:1000px}
.algo-head{background:#0b1f3a;color:#ffffff;padding:7px 12px;font-weight:600;font-size:13px;letter-spacing:.2px}
.algo-head b{color:#c9a227}
.algo-body{padding:8px 12px;font-size:13px;line-height:1.45;color:#1b1b1b}
.algo-box table{border-collapse:collapse;width:100%;font-size:12.5px}
.algo-box th{background:#f3efe1;color:#0b1f3a;text-align:left;padding:5px 8px;border-bottom:2px solid #c9a227}
.algo-box td{padding:5px 8px;border-bottom:1px solid #e6e6e6;vertical-align:top;color:#1b1b1b}
.algo-err{border-color:#7a1020;border-left-color:#7a1020}
.algo-err .algo-head{background:#7a1020}
.algo-ok{color:#1d6b35;font-weight:600}.algo-warn{color:#9a5b00;font-weight:600}.algo-bad{color:#a01020;font-weight:600}
.algo-box code{background:#f4f4f4;padding:1px 4px;border-radius:3px;color:#0b1f3a}
.algo-lanes{display:flex;gap:10px;align-items:flex-start;overflow-x:auto;padding:4px 0}
.algo-lane{display:flex;flex-direction:column;gap:8px;min-width:112px;max-width:150px;flex:1 1 0}
.algo-node{border-radius:6px;padding:6px 9px;font-size:12px}
.algo-node small{display:block;opacity:.8;font-size:10.5px;margin-top:2px}
.n-input{background:#f3efe1;color:#0b1f3a;border:1px solid #c9a227}
.n-agent{background:#0b1f3a;color:#fff}
.n-par{background:#1d3b66;color:#fff;border:2px solid #c9a227}
.n-cond{background:#c9a227;color:#0b1f3a;font-weight:600}
.n-branch{background:#fff;color:#0b1f3a;border:1.5px dashed #0b1f3a}
.n-emit{background:#fff;color:#0b1f3a;border:2px solid #c9a227;font-weight:600}
.n-op{background:#eef1f6;color:#0b1f3a}
</style>"""


def _box(title, body, kind="ok"):
    cls = "algo-box algo-err" if kind == "err" else "algo-box"
    display(HTML(f'{CSS}<div class="{cls}"><div class="algo-head">{title}</div>'
                 f'<div class="algo-body">{body}</div></div>'))


def _esc(x, n=None):
    s = str(x)
    if n and len(s) > n:
        s = s[:n] + "…"
    return html.escape(s)


def _fmt_value(v):
    v = algoruntime.Runtime.value(v)
    if isinstance(v, list):
        items = "".join(f"<li>{_esc(algoruntime.Runtime.value(x))}</li>" for x in v)
        return f"<ol style='margin:0;padding-left:18px'>{items}</ol>"
    return _esc(v)


# ─────────────────────────── the magics ───────────────────────────
@magics_class
class AlgoMagics(Magics):
    def __init__(self, shell):
        super().__init__(shell)
        self.chunks: list[tuple[set, str]] = []   # (names defined, source) from --define cells

    # helpers ----------------------------------------------------------
    @staticmethod
    def _names(src):
        tree = PARSER.parse(src)
        return {str(it.children[0]) for it in tree.children
                if getattr(it, "data", "") in ("agent_decl", "flow_decl")}

    def _compile(self, chunks, cell, name, define=False):
        prefix = "\n".join(s for _, s in chunks)
        src = prefix + ("\n" if prefix else "") + cell
        off = prefix.count("\n") + (1 if prefix else 0)
        try:
            code, prog = compile_source(src, f"{name}.algo")
            return code, prog, [], off
        except CompileError as e:
            diags = [d for d in e.diags if not (define and d.code == "E111")]
            return None, None, diags, off

    def _show_diags(self, diags, off):
        rows = []
        for d in diags:
            ln = d.line - off
            where = f"line {ln}" if ln >= 1 else "an earlier <code>--define</code> cell"
            msg = _esc(d.msg).replace("\n", "<br>")
            rows.append(f"<tr><td><b>{d.code}</b></td><td>{where}</td><td>{msg}</td></tr>")
        _box(f"✗ <b>{len(diags)}</b> compile error(s) — nothing was run",
             f"<table><tr><th>Code</th><th>Where</th><th>Problem</th></tr>{''.join(rows)}</table>", "err")

    def _parse_line(self, line, run=True):
        ap = argparse.ArgumentParser(prog="%%algo", add_help=False)
        ap.add_argument("--define", action="store_true")
        ap.add_argument("--check", action="store_true")
        ap.add_argument("--arg", action="append", default=[])
        ap.add_argument("--live", action="store_true")
        ap.add_argument("--interactive", action="store_true")
        ap.add_argument("--graph", action="store_true")
        ap.add_argument("--python", action="store_true")
        ap.add_argument("--name", default="notebook")
        try:
            a, rest = ap.parse_known_args(shlex.split(line))
        except SystemExit:
            _box("✗ could not read the options", "Example: <code>%%algo --arg \"topic=AI ethics\" --graph</code>", "err")
            return None, []
        return a, rest

    # %%algo ------------------------------------------------------------
    @cell_magic
    def algo(self, line, cell):
        a, rest = self._parse_line(line)
        if a is None:
            return
        if rest:
            _box("✗ unknown option", f"<code>{_esc(' '.join(rest))}</code> — see <code>%algo_help</code>", "err")
            return
        if a.define:
            return self._define(cell)
        self._run(cell, a, chunks=self.chunks)

    def _define(self, cell):
        try:
            names = self._names(cell)
        except Exception:  # noqa — let the compiler produce the proper E100 message
            names = set()
        kept = [c for c in self.chunks if not (c[0] & names)]
        _, _, diags, off = self._compile(kept, cell, "notebook", define=True)
        if diags:
            return self._show_diags(diags, off)
        replaced = len(self.chunks) - len(kept)
        self.chunks = kept + [(names, cell)]
        total = set().union(*(c[0] for c in self.chunks)) if self.chunks else set()
        what = ", ".join(f"<code>{n}</code>" for n in sorted(names)) or "models / tools"
        note = " (replaced the earlier definition)" if replaced else ""
        _box(f"✓ defined {what}{note}",
             f"Session now has {len(total)} agent(s)/flow(s). Run a cell with <code>flow main</code> to use them.")

    def _run(self, cell, a, chunks):
        code, prog, diags, off = self._compile(chunks, cell, a.name)
        if diags:
            return self._show_diags(diags, off)
        if a.check:
            return _box("✓ no errors",
                        f"{len(prog.agents)} agent(s) · {len(prog.flows)} flow(s) · {len(prog.tools)} tool(s)")
        if a.python:
            display(Markdown(f"**Generated Python**\n\n```python\n{code}\n```"))
        if a.graph:
            self._graph(prog)

        main = prog.flows["main"]
        vals = {}
        for kv in a.arg:
            if "=" not in kv:
                return _box("✗ bad --arg", f"Use <code>--arg \"name=value\"</code>; got <code>{_esc(kv)}</code>", "err")
            k, v = kv.split("=", 1)
            vals[k.strip()] = v
        for p in main.params:
            if p.name not in vals:
                vals[p.name] = input(f"{p.name}: ")
        unknown = set(vals) - {p.name for p in main.params}
        if unknown:
            return _box("✗ unknown input", f"flow main has no input {', '.join(map(_esc, unknown))}", "err")

        os.environ["ALGO_HOME"] = str(HERE)
        os.environ["ALGO_NOTEBOOK"] = "1"     # runtime prints progress only; the table shows results
        os.environ.setdefault("ALGO_AUDIT_DIR", "runs")
        for var, on in (("ALGO_LIVE", a.live), ("ALGO_INTERACTIVE", a.interactive)):
            if on:
                os.environ[var] = "1"
            else:
                os.environ.pop(var, None)

        ns = {"__name__": "algo_notebook"}
        try:
            exec(compile(code, f"<algo:{a.name}>", "exec"), ns)
        except SystemExit as e:   # e.g. --live without a key
            return _box("✗ could not start", _esc(e.code), "err")

        rt, result, problem = ns["rt"], None, None
        try:
            result = ns["flow_main"](**vals)
        except algoruntime.PolicyViolation as e:
            problem = ("✗ policy violation — run stopped", _esc(e))
        except Exception as e:  # noqa
            problem = ("✗ an agent call failed", f"{_esc(type(e).__name__)}: {_esc(e, 400)}")

        self.shell.user_ns["algo_result"] = result
        self.shell.user_ns["algo_calls"] = rt.calls
        self._summary(rt, result, problem)

    def _summary(self, rt, result, problem):
        rows = []
        for c in rt.calls:
            st = c.get("status")
            cls = {"ok": "algo-ok", "approved": "algo-ok", "escalated": "algo-warn"}.get(st, "algo-bad")
            label = {"escalated": "⚠ human review", "approved": "✔ approved", "ok": "ok"}.get(st, st)
            conf = f"{c['confidence']:.2f}" if "confidence" in c else "—"
            out = c.get("output")
            out = _fmt_value(out) if out is not None else _esc(c.get("reason") or c.get("error") or "")
            rows.append(f"<tr><td><b>{_esc(c.get('agent'))}</b></td>"
                        f"<td style='white-space:nowrap'>{_esc(c.get('model', '—'))}</td>"
                        f"<td>{conf}</td><td class='{cls}'>{_esc(label)}</td><td>{c.get('tokens', '—')}</td>"
                        f"<td>{out}</td></tr>")
        s = rt.stats
        body = (f"<table><tr><th>Agent</th><th>Model</th><th>Conf.</th><th>Status</th><th>Tokens</th>"
                f"<th>Output</th></tr>{''.join(rows)}</table>")
        if problem:
            body = f"<p class='algo-bad'>{problem[1]}</p>" + body
        elif result is not None:
            emitted = "".join(f"<li style='margin-bottom:4px'>{_fmt_value(v)}</li>" for v in result)
            body += f"<p style='margin:10px 0 4px'><b>Emitted</b> (also in <code>algo_result</code>)</p><ol>{emitted}</ol>"
        head = problem[0] if problem else "✓ run complete"
        head += (f" · <b>{s['calls']}</b> agent calls · ~{s['tokens']} tokens · {s['escalations']}"
                 f" escalated · backend <b>{rt.backend.name}</b>")
        _box(head, body + f"<p style='margin:8px 0 0;font-size:11.5px;color:#555'>audit log: "
                          f"<code>{_esc(rt.audit_path)}</code> · run {rt.run_id}</p>",
             "err" if problem else "ok")

    # flow diagram -----------------------------------------------------
    def _graph(self, prog):
        nodes, edges = flow_graph(prog)

        if shutil.which("dot"):
            try:
                import graphviz
                style = {
                    "input": dict(shape="oval", style="filled", fillcolor="#f3efe1", color="#c9a227", fontcolor="#0b1f3a"),
                    "agent": dict(shape="box", style="rounded,filled", fillcolor="#0b1f3a", fontcolor="white", color="#0b1f3a"),
                    "par": dict(shape="box", style="rounded,filled", fillcolor="#1d3b66", fontcolor="white", color="#c9a227", penwidth="2"),
                    "cond": dict(shape="diamond", style="filled", fillcolor="#c9a227", fontcolor="#0b1f3a", color="#c9a227"),
                    "branch": dict(shape="box", style="rounded,dashed", fontcolor="#0b1f3a", color="#0b1f3a"),
                    "emit": dict(shape="oval", style="bold", color="#c9a227", fontcolor="#0b1f3a"),
                }
                g = graphviz.Digraph(graph_attr=dict(rankdir="LR", bgcolor="white", fontname="Helvetica", pad="0.2"),
                                     node_attr=dict(fontname="Helvetica", fontsize="11"),
                                     edge_attr=dict(color="#0b1f3a", arrowsize="0.7"))
                for nid, label, kind, sub in nodes:
                    g.node(nid, f"{label}\n{sub}" if sub else label, **style[kind])
                for a, b, dashed in edges:
                    g.edge(a, b, style="dashed" if dashed else "solid")
                svg = g.pipe(format="svg").decode()
                svg = svg[svg.index("<svg"):]                      # drop XML prolog / doctype
                svg = re.sub(r'<svg width="[^"]*" height="[^"]*"',
                             '<svg style="width:100%;max-width:960px;height:auto"', svg, count=1)
                display(HTML(f"{CSS}<div class='algo-box'><div class='algo-head'>flow <b>main</b> — agent graph</div>"
                             f"<div class='algo-body'>{svg}</div></div>"))
                return
            except Exception:  # noqa — fall back to the HTML layout
                pass
        # dependency-free fallback: columns by depth, each node lists what feeds it
        depth = {}
        for nid, *_ in nodes:
            ins = [a for a, b, _ in edges if b == nid]
            depth[nid] = 1 + max((depth.get(a, 0) for a in ins), default=-1)
        label_of = {nid: label for nid, label, *_ in nodes}
        lanes = {}
        for nid, label, kind, sub in nodes:
            ins = [label_of[a] for a, b, _ in edges if b == nid]
            feeds = f"<small>← {_esc(', '.join(dict.fromkeys(ins)))}</small>" if ins else ""
            subl = f"<small>{_esc(sub)}</small>" if sub else ""
            lanes.setdefault(depth[nid], []).append(f"<div class='algo-node n-{kind}'>{_esc(label)}{subl}{feeds}</div>")
        cols = "".join(f"<div class='algo-lane'>{''.join(v)}</div>" for _, v in sorted(lanes.items()))
        _box("flow <b>main</b> — agent graph (left → right)", f"<div class='algo-lanes'>{cols}</div>")

    # line magics -------------------------------------------------------
    @line_magic
    def algo_run(self, line):
        parts = shlex.split(line)
        if not parts:
            return _box("✗ which file?", "Example: <code>%algo_run examples/research_brief.algo --arg \"topic=AI\"</code>", "err")
        path = Path(parts[0])
        if not path.exists() and (HERE / path).exists():
            path = HERE / path
        if not path.exists():
            return _box("✗ file not found", _esc(parts[0]), "err")
        a, rest = self._parse_line(shlex.join(parts[1:]))
        if a is None:
            return
        a.name = path.stem
        self._run(path.read_text(encoding="utf-8"), a, chunks=[])

    @line_magic
    def algo_list(self, line):
        if not self.chunks:
            return _box("session is empty", "Use <code>%%algo --define</code> to add agents.")
        rows = "".join(f"<tr><td>{i + 1}</td><td>{', '.join(sorted(n)) or '(models / tools)'}</td>"
                       f"<td>{s.count(chr(10)) + 1} lines</td></tr>" for i, (n, s) in enumerate(self.chunks))
        _box("session definitions", f"<table><tr><th>#</th><th>Defines</th><th>Size</th></tr>{rows}</table>")

    @line_magic
    def algo_reset(self, line):
        self.chunks = []
        _box("✓ session cleared", "All <code>--define</code> cells were forgotten.")

    @line_magic
    def algo_help(self, line):
        display(Markdown(f"```text\n{__doc__}\n```"))


def load_ipython_extension(ip):
    ip.register_magics(AlgoMagics)
    _box(f"AlgoLang <b>{VERSION}</b> ready",
         "Write AlgoLang in any cell that starts with <code>%%algo</code>. "
         "Options: <code>--arg \"name=value\"</code> · <code>--define</code> · <code>--graph</code> · "
         "<code>--check</code> · <code>--python</code> · <code>--interactive</code> · <code>--live</code>. "
         "Help: <code>%algo_help</code>")


Overwriting algokit/algomagic.py


**Enterprise CLI helpers** — `algolang/algocli_ext.py` (collapsed; double-click to view)

In [6]:
%%writefile algokit/algocli_ext.py
"""
algocli_ext — enterprise commands for the `algo` command line (AlgoLang v0.2)
AlgoProfessor AI R&D Solutions

    algo init [DIR]                       scaffold an enterprise project (config, tools, knowledge base)
    algo domains [QUERY] [--sector S]     browse or search 120 domain packs in 20 sectors
    algo pack ID [--copy]                 show a domain pack, or copy it here to customise
    algo batch PROGRAM --input data.csv   run a program over every row, in parallel
    algo review [approve|reject ID]       the human review queue
    algo audit verify FILE|DIR            check the tamper-evident audit chain
    algo report [DIR]                     usage, cost and governance report
    algo chat                             prompt-based mode: describe the problem in plain English
"""
from __future__ import annotations

import json
import os
import shutil
import sys
import textwrap
import time
from pathlib import Path

HERE = Path(__file__).resolve().parent
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

import algolang as al  # noqa: E402


def resolve_program(arg: str) -> Path | None:
    """A .algo path, or pack:<id> / a bare pack id from the library."""
    if arg.startswith("pack:"):
        arg = arg[5:]
    p = Path(arg)
    if p.exists():
        return p
    try:
        return HERE / al.domains.get(arg)["path"]
    except KeyError:
        return None


def _table(rows, headers, widths=None):
    widths = widths or [max(len(str(h)), *(len(str(r[i])) for r in rows)) for i, h in enumerate(headers)] if rows else \
        [len(h) for h in headers]
    line = "  " + "  ".join(str(h).ljust(w) for h, w in zip(headers, widths))
    print(line)
    print("  " + "  ".join("─" * w for w in widths))
    for r in rows:
        print("  " + "  ".join(str(c)[:w].ljust(w) for c, w in zip(r, widths)))


# ─────────────────────────────── init ───────────────────────────────
CONFIG_TEMPLATE = {
    "organisation": "My Organisation",
    "policy": {
        "forbid": ["gender", "caste", "religion"],
        "max_budget_tokens": 8000,
        "min_human_below": 0.6,
        "allowed_tools": ["kb.search", "calc.eval", "file.read", "company.directory"],
    },
    "max_concurrency": 16,
    "max_run_tokens": 200000,
    "cache": {"enabled": True},
    "audit_dir": "runs",
    "models": {"smart": "claude-sonnet-5-5", "fast": "claude-haiku-5-5"},
    "tools": {"file.read": {"root": "data"}, "kb.search": {"dir": "kb"}},
    "plugins": ["plugins/company_tools.py"],
}

PLUGIN_TEMPLATE = '''"""Company tools for AlgoLang agents. Agents get a tool only if they `uses` it
and it is in policy.allowed_tools in algo.config.json."""
from algoruntime import register_tool

DIRECTORY = {"finance": "finance-desk@example.com", "legal": "legal-desk@example.com",
             "it": "it-helpdesk@example.com", "hr": "people-team@example.com"}


@register_tool("company.directory", "Find the right internal team and contact for a request")
def company_directory(query: str) -> str:
    q = query.lower()
    hits = [f"{team}: {contact}" for team, contact in DIRECTORY.items() if team in q]
    return "; ".join(hits) or "no matching team — route to it-helpdesk@example.com"
'''

KB_TEMPLATE = """Refund policy

Customers may request a refund within 30 days of purchase. Refunds above 50,000 INR need manager approval.

Data protection

Personal data is kept for 7 years and is never shared without consent.
"""

PROGRAM_TEMPLATE = """// Combine two domain packs from the library and add your own agent.
use pack insurance.claims_triage as claims
use pack legal.compliance_gap as compliance
tool company.directory

agent Router(summary: text) -> text {
    role "Operations coordinator"
    goal "Decide which internal team should own: {summary}"
    uses company.directory
}

flow main(claim: text) {
    intake  = claims.Intake(claim)
    covered = claims.Coverage(intake)
    level   = claims.Complexity(covered)
    owner   = Router(covered)
    emit covered, level, owner
}
"""


def cmd_init(directory="."):
    d = Path(directory)
    (d / "plugins").mkdir(parents=True, exist_ok=True)
    (d / "kb").mkdir(exist_ok=True)
    (d / "data").mkdir(exist_ok=True)
    (d / "programs").mkdir(exist_ok=True)
    made = []
    for path, content in ((d / "algo.config.json", json.dumps(CONFIG_TEMPLATE, indent=2)),
                          (d / "plugins" / "company_tools.py", PLUGIN_TEMPLATE),
                          (d / "kb" / "policies.md", KB_TEMPLATE),
                          (d / "programs" / "claims_ops.algo", PROGRAM_TEMPLATE),
                          (d / "data" / "claims.csv", "claim\nRear-end collision in Pune, bumper damage\n"
                                                     "Water damage after pipe burst\nLaptop stolen from parked car\n")):
        if not path.exists():
            path.write_text(content, encoding="utf-8")
            made.append(path)
    print(f"✓ enterprise project ready in {d.resolve()}")
    for p in made:
        print(f"    {p}")
    print("\n  next:")
    print(f"    cd {d}")
    print("    algo run programs/claims_ops.algo --arg \"claim=Rear-end collision\"")
    print("    algo batch programs/claims_ops.algo --input data/claims.csv")
    print("    algo report")
    return 0


# ─────────────────────────────── domains / pack ───────────────────────────────
def cmd_domains(query=None, sector=None, limit=10):
    if query:
        res = al.domains.search(query, k=limit)
        res = [r for r in res if r["score"] >= 0.3 * res[0]["score"]] if res else res
        if not res:
            print("  no matching domain packs")
            return 1
        _table([(r["id"], r["title"], r["pattern"], r["risk"], r["score"]) for r in res],
               ["pack", "title", "pattern", "risk", "match"])
        print(f"\n  run one:  algo run pack:{res[0]['id']} --arg \"{res[0]['input']}=...\"")
        return 0
    if sector:
        packs = al.domains.all(sector)
        if not packs:
            print(f"  unknown sector '{sector}' — run `algo domains` to list sectors")
            return 1
        _table([(p["id"], p["title"], p["pattern"], p["risk"], "yes" if p["people"] else "") for p in packs],
               ["pack", "title", "pattern", "risk", "people"])
        return 0
    secs = al.domains.sectors()
    print(f"  AlgoLang domain library — {sum(s['packs'] for s in secs)} packs in {len(secs)} sectors\n")
    _table([(s["id"], s["name"], s["packs"]) for s in secs], ["sector", "name", "packs"])
    print('\n  algo domains --sector insurance      ·      algo domains "screen CVs fairly"')
    return 0


def cmd_pack(pack_id, copy=False):
    try:
        m = al.domains.get(pack_id)
    except KeyError as e:
        print(f"✗ {e.args[0]}")
        return 1
    src = HERE / m["path"]
    print(f"  {m['id']} — {m['title']}  ({m['sector_name']})")
    print(f"  {m['description']}")
    print(f"  pattern {m['pattern']} · risk {m['risk']} · human review below {m['human_threshold']} · "
          f"budget {m['budget_tokens']} tokens" + (f" · forbids {', '.join(m['forbid'])}" if m["forbid"] else ""))
    print(f"  agents: {', '.join(m['agents'])}" + (f" · tools: {', '.join(m['tools'])}" if m["tools"] else ""))
    if copy:
        dest = Path.cwd() / f"{m['id'].split('.')[1]}.algo"
        shutil.copy(src, dest)
        print(f"\n✓ copied to {dest} — edit it, then: algo run {dest.name} --arg \"{m['input']}=...\"")
    else:
        print(f"\n  use it:   use pack {m['id']}            (in your own .algo file)")
        print(f"  run it:   algo run pack:{m['id']} --arg \"{m['input']}=...\"")
        print(f"  edit it:  algo pack {m['id']} --copy")
    return 0


# ─────────────────────────────── batch ───────────────────────────────
def cmd_batch(program, input_csv, out=None, workers=8, live=False):
    path = resolve_program(program)
    if not path:
        print(f"✗ not a file or domain pack: {program}")
        return 1
    try:
        prog = al.load(path)
    except al.CompileError as e:
        print(f"✗ {path.name}: {len(e.diags)} error(s)")
        for d in e.diags:
            print(f"  {d}")
        return 1
    out = out or str(Path(input_csv).with_name(Path(input_csv).stem + "_results.csv"))
    t0 = time.time()
    last = [0.0]

    def progress(done, total):
        if time.time() - last[0] > 0.25 or done == total:
            last[0] = time.time()
            bar = "█" * int(30 * done / total)
            print(f"\r  [{bar:<30}] {done}/{total}", end="", flush=True)

    print(f"▶ batch {prog.name} · inputs {prog.inputs} · {workers} workers · backend {'live' if live else 'mock'}")
    try:
        results = al.batch_csv(prog, input_csv, out, workers=workers, live=live, progress=progress)
    except (ValueError, FileNotFoundError) as e:
        print(f"\n✗ {e}")
        return 1
    print()
    st = {}
    for r in results:
        st[r.status] = st.get(r.status, 0) + 1
    esc = sum(1 for r in results if r.escalated)
    toks = sum(r.stats.get("tokens", 0) for r in results)
    print(f"■ {len(results)} rows in {time.time() - t0:.1f}s · " + " · ".join(f"{v} {k}" for k, v in st.items())
          + f" · {esc} rows sent for human review · ~{toks} tokens")
    print(f"  results → {out}")
    if esc:
        print("  review them:  algo review")
    return 0 if st.get("failed", 0) == 0 else 3


# ─────────────────────────────── review ───────────────────────────────
def cmd_review(action=None, item=None, reviewer="", note="", show_all=False):
    if action in ("approve", "reject"):
        if not item:
            print("  usage: algo review approve <id> --reviewer \"Name\" --note \"why\"")
            return 1
        try:
            full = al.review.approve(item, reviewer, note) if action == "approve" else al.review.reject(item, reviewer, note)
        except KeyError as e:
            print(f"✗ {e.args[0]}")
            return 1
        print(f"✓ {action}d {full}" + (f" by {reviewer}" if reviewer else ""))
        return 0
    items = al.review.all() if show_all else al.review.pending()
    if not items:
        print("  review queue is empty" if not show_all else "  no review items yet")
        return 0
    print(f"  {len(items)} item(s){' (all)' if show_all else ' waiting for a human decision'}\n")
    _table([(i["id"][:8], i["program"], i["agent"], f"{i['confidence']:.2f}<{i['threshold']:.2f}",
             i["status"], i["output"][:40]) for i in items[-30:]],
           ["id", "program", "agent", "confidence", "status", "output"])
    if len(items) > 30:
        print(f"  … {len(items) - 30} older item(s) not shown")
    print('\n  algo review approve <id> --reviewer "Name" --note "checked"   ·   algo review reject <id>')
    return 0


# ─────────────────────────────── audit / report ───────────────────────────────
def cmd_audit_verify(target):
    t = Path(target)
    files = sorted(t.glob("*.jsonl")) if t.is_dir() else [t]
    if not files or not files[0].exists():
        print(f"✗ nothing to verify at {t}")
        return 1
    bad = 0
    for f in files:
        r = al.audit.verify(f)
        if r["ok"]:
            print(f"  ✓ {f.name:<36} {r['records']} records, chain intact")
        else:
            bad += 1
            print(f"  ✗ {f.name:<36} line {r['first_bad_line']}: {r['reason']}")
    return 0 if not bad else 4


def cmd_report(directory="runs", as_json=False):
    rep = al.audit.report(directory)
    if as_json:
        print(json.dumps(rep, indent=2))
        return 0
    t = rep["totals"]
    print(f"  AlgoLang usage & governance report — {directory}\n")
    print(f"  {t['calls']} agent calls · ~{t['tokens']} tokens · {t['escalated']} escalated · "
          f"{t['blocked']} blocked · {t['failed']} failed · {t['cached']} from cache · {t['tool_calls']} tool calls\n")
    if rep["programs"]:
        _table([(k, v["calls"], v["tokens"], v["escalated"], v["blocked"], v["avg_confidence"])
                for k, v in sorted(rep["programs"].items(), key=lambda kv: -kv[1]["tokens"])],
               ["program", "calls", "tokens", "escalated", "blocked", "avg conf"])
        print()
        _table([(k, v["calls"], v["tokens"], v["escalated"], v["avg_confidence"])
                for k, v in sorted(rep["agents"].items(), key=lambda kv: -kv[1]["tokens"])[:12]],
               ["top agents by tokens", "calls", "tokens", "escalated", "avg conf"])
    broken = [k for k, ok in rep["chains"].items() if not ok]
    print(f"\n  audit chains: {len(rep['chains']) - len(broken)} intact" + (f", {len(broken)} BROKEN: {', '.join(broken)}" if broken else ""))
    return 0


# ─────────────────────────────── chat (prompt mode) ───────────────────────────────
CHAT_HELP = """  Describe a problem in plain English — AlgoLang finds the right domain pack and runs it.
    1, 2, 3        switch to one of the alternatives shown
    :show          show the program   ·   :save NAME   save it to customise
    :ai CHANGE     ask AI to change the program (needs ANTHROPIC_API_KEY)
    :live on|off   real Claude models / offline mock   ·   :help   ·   :quit"""


def _print_run(res, prog):
    for c in res.calls:
        flag = "  ⚠ human review" if c.get("status") == "escalated" else ""
        if c.get("status") == "blocked":
            print(f"    ✗ {c['agent']}: blocked — {c.get('reason')}")
            continue
        out = c.get("output")
        out = f"[{len(out)} items]" if isinstance(out, list) else str(out)
        print(f"    → {c['agent']:<20} conf {c.get('confidence', 0):.2f}  {out[:70]}{flag}")
    if res.ok:
        print("  result:")
        for v in res.values:
            v = al._rt.Runtime.value(v)
            for line in textwrap.wrap(str(v), 96)[:4]:
                print(f"    {line}")
    else:
        print(f"  ✗ {res.problem}")
    s = res.stats
    print(f"  {s.get('calls', 0)} agent calls · ~{s.get('tokens', 0)} tokens · {s.get('escalations', 0)} sent for review"
          f" · audit {res.audit_path}")


def cmd_chat():
    print(f"AlgoLang {al.__version__} — prompt mode ({len(al.domains.all())} domain packs)")
    print(CHAT_HELP + "\n")
    live, current, alts, last_request, source = False, None, [], "", None
    while True:
        try:
            line = input("you> ").strip()
        except (EOFError, KeyboardInterrupt):
            print()
            return 0
        if not line:
            continue
        if line in (":quit", ":q", ":exit"):
            return 0
        if line == ":help":
            print(CHAT_HELP)
            continue
        if line.startswith(":live"):
            live = line.endswith("on")
            print(f"  backend: {'live (Claude)' if live else 'mock (offline)'}")
            continue
        if line == ":show":
            print(source or "  (nothing yet — describe a problem first)")
            continue
        if line.startswith(":save"):
            if not source:
                print("  nothing to save yet")
                continue
            name = (line.split(maxsplit=1)[1:] or ["my_program"])[0]
            name = name if name.endswith(".algo") else name + ".algo"
            Path(name).write_text(source, encoding="utf-8")
            print(f"  ✓ saved {name} — run it with: algo run {name}")
            continue
        if line.startswith(":ai"):
            change = line[3:].strip()
            if not (source and change):
                print("  usage: :ai make the reviewer stricter and add a Hindi summary")
                continue
            try:
                r = al.ask(change, current=source, log=lambda m: print("  " + m))
            except Exception as e:  # noqa
                print(f"  ✗ {e}")
                continue
            if r["ok"]:
                source, current = r["source"], al.compile(r["source"], "ai_program")
                print("  ✓ program updated — running it")
                inp = {current.inputs[0]: last_request} if len(current.inputs) == 1 else {}
                if len(inp) == len(current.inputs):
                    _print_run(current.run(live=live, **inp), current)
            continue
        if line.isdigit() and alts and 1 <= int(line) <= len(alts):
            pick = alts[int(line) - 1]
            alts = [current_meta] + [a for a in alts if a["id"] != pick["id"]]  # noqa: F821
            current_meta = pick
        else:
            last_request = line
            matches = al.domains.search(line, k=4)
            if not matches:
                print("  no domain pack matches that — try describing the task differently, or use :ai")
                continue
            current_meta, alts = matches[0], matches[1:]
        current = al.load(HERE / current_meta["path"])
        source = current.source
        print(f"  ▶ {current_meta['title']} ({current_meta['id']}) · {current_meta['pattern']} pattern · "
              f"risk {current_meta['risk']} · agents: {', '.join(current_meta['agents'])}")
        if alts:
            print("    other options: " + "  ".join(f"[{n}] {a['title']}" for n, a in enumerate(alts, 1)))
        inputs = {current.inputs[0]: last_request}
        _print_run(current.run(live=live, **inputs), current)


Overwriting algokit/algocli_ext.py


**Python SDK** — `algolang/algolang/__init__.py` (collapsed; double-click to view)

In [7]:
%%writefile algokit/algolang/__init__.py
"""
AlgoLang SDK  (v0.2 "Enterprise")
AlgoProfessor AI R&D Solutions

The one Python API behind all three ways of working — the `algo` command line, AlgoLang Studio
(graphical) and the Prompt UI. Use it to embed governed multi-agent programs in your own systems.

    import algolang as al

    prog = al.load("claims.algo")                     # or al.compile(source) or al.pack("insurance.claims_triage")
    res  = prog.run(claim="Rear-end collision, Pune")  # RunResult: values, calls, stats, ok, problem
    rows = prog.run_batch([{"claim": c} for c in claims], workers=16)

    al.domains.search("triage insurance claims")       # 120 domain packs in 20 sectors
    al.compose("check this NDA against our playbook")  # prompt → best domain pack → ready program

    @al.tool("crm.lookup", "Look up a customer by name or ID")
    def crm_lookup(query: str) -> str: ...

    al.configure(policy={"forbid": ["salary"], "min_human_below": 0.8}, max_concurrency=32)
    al.audit.verify("runs/claims.audit.jsonl");  al.audit.report("runs")
    al.review.pending();  al.review.approve(item_id, reviewer="Dr. S")
"""
from __future__ import annotations

import builtins
import csv
import json
import math
import os
import re
import sys
import threading
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, field
from pathlib import Path

HOME = Path(__file__).resolve().parents[1]
if str(HOME) not in sys.path:
    sys.path.insert(0, str(HOME))
os.environ.setdefault("ALGO_HOME", str(HOME))

import algoruntime as _rt  # noqa: E402
from algoc import CompileError, Diag, compile_source, flow_graph  # noqa: E402

__version__ = _rt.VERSION
__all__ = ["compile", "load", "pack", "Program", "RunResult", "CompileError", "tool", "configure",
           "domains", "compose", "ask", "audit", "review", "batch_csv"]

PolicyViolation = _rt.PolicyViolation
configure = _rt.configure
tool = _rt.register_tool


# ══════════════════════════════ programs ═════════════════════════════════════
@dataclass
class RunResult:
    values: list = field(default_factory=list)
    calls: list = field(default_factory=list)
    stats: dict = field(default_factory=dict)
    run_id: str = ""
    ok: bool = True
    problem: str | None = None          # policy violation or failure message
    status: str = "ok"                  # ok | blocked | failed
    inputs: dict = field(default_factory=dict)
    audit_path: str = ""

    @property
    def escalated(self):
        return [c for c in self.calls if c.get("status") == "escalated"]

    def to_dict(self):
        def j(v):
            v = _rt.Runtime.value(v)
            return [j(x) for x in v] if isinstance(v, list) else (v if isinstance(v, (str, int, float, bool, type(None))) else str(v))
        return dict(run_id=self.run_id, ok=self.ok, status=self.status, problem=self.problem, inputs=self.inputs,
                    values=[j(v) for v in self.values], stats=self.stats, audit_path=self.audit_path,
                    calls=[{k: (j(v) if k == "output" else v) for k, v in c.items()} for c in self.calls])


class Program:
    """A compiled, checked AlgoLang program."""

    def __init__(self, source: str, name: str = "program", base_dir=None):
        self.source, self.name = source, re.sub(r"\W+", "_", Path(name).stem) or "program"
        self.base_dir = Path(base_dir) if base_dir else Path.cwd()
        self.python, self._prog = compile_source(source, f"{self.name}.algo", self.base_dir)
        self._code = builtins.compile(self.python, f"<algolang:{self.name}>", "exec")

    # facts about the program
    @property
    def inputs(self):
        return [p.name for p in self._prog.flows["main"].params]

    @property
    def agents(self):
        return sorted(self._prog.agents)

    @property
    def tools(self):
        return sorted(self._prog.tools)

    @property
    def imports(self):
        return list(self._prog.imports)

    def policies(self):
        return {n: a.policy for n, a in self._prog.agents.items()}

    def graph(self):
        nodes, edges = flow_graph(self._prog)
        return {"nodes": [dict(id=a, label=b, kind=c, sub=d) for a, b, c, d in nodes],
                "edges": [dict(src=a, dst=b, dashed=c) for a, b, c in edges]}

    # running
    def run(self, live: bool = False, quiet: bool = True, audit_dir: str | None = None, backend=None,
            **inputs) -> RunResult:
        missing = [p for p in self.inputs if p not in inputs]
        if missing:
            raise TypeError(f"{self.name}: missing input(s): {', '.join(missing)}")
        ns = {"__name__": "algolang_sdk",
              "__algo_runtime__": dict(quiet=quiet, live=live, backend=backend, audit_dir_override=audit_dir)}
        exec(self._code, ns)
        rt = ns["rt"]
        res = RunResult(run_id=rt.run_id, inputs={k: inputs[k] for k in self.inputs},
                        audit_path=str(rt.audit_path))
        try:
            res.values = ns["flow_main"](**{k: str(inputs[k]) for k in self.inputs}) or []
        except _rt.PolicyViolation as e:
            res.ok, res.status, res.problem = False, "blocked", str(e)
        except Exception as e:  # noqa
            res.ok, res.status, res.problem = False, "failed", f"{type(e).__name__}: {e}"
        res.calls, res.stats = list(rt.calls), dict(rt.stats)
        return res

    def run_batch(self, rows, workers: int = 8, live: bool = False, audit_dir: str | None = None,
                  progress=None, quiet: bool = True) -> list[RunResult]:
        """Run the flow once per row (dict of inputs), in parallel. Model calls stay within max_concurrency."""
        rows = list(rows)
        out = [None] * len(rows)
        done = [0]
        lock = threading.Lock()

        def one(i):
            out[i] = self.run(live=live, quiet=quiet, audit_dir=audit_dir, **rows[i])
            with lock:
                done[0] += 1
                if progress:
                    progress(done[0], len(rows))

        with ThreadPoolExecutor(max_workers=max(1, workers)) as ex:
            list(ex.map(one, range(len(rows))))
        return out

    def __repr__(self):
        return f"<AlgoLang Program {self.name}: {len(self.agents)} agents, inputs={self.inputs}>"


def compile(source: str, name: str = "program", base_dir=None) -> Program:  # noqa: A001
    """Compile AlgoLang source. Raises CompileError with every problem found."""
    return Program(source, name, base_dir)


def load(path) -> Program:
    p = Path(path)
    return Program(p.read_text(encoding="utf-8"), p.stem, p.resolve().parent)


def pack(pack_id: str) -> Program:
    """Load a domain pack from the library, e.g. al.pack("insurance.claims_triage")."""
    meta = domains.get(pack_id)
    return load(HOME / meta["path"])


def batch_csv(program: Program, in_path, out_path, workers: int = 8, live=False, progress=None):
    """Run a program over every row of a CSV file and write results to another CSV.

    Columns whose names match the flow's inputs are used; with one input and no matching column,
    the first column is used."""
    with open(in_path, newline="", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        cols, raw = reader.fieldnames or [], list(reader)
    rows = []
    for r in raw:
        if all(i in r for i in program.inputs):
            rows.append({i: r[i] for i in program.inputs})
        elif len(program.inputs) == 1 and cols:
            rows.append({program.inputs[0]: r[cols[0]]})
        else:
            raise ValueError(f"CSV needs columns named {', '.join(program.inputs)}")
    results = program.run_batch(rows, workers=workers, live=live, progress=progress)
    width = max((len(r.values) for r in results), default=0)
    with open(out_path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["row", *program.inputs, "status", "escalated", "tokens", "run_id", "problem",
                    *[f"output_{i + 1}" for i in range(width)]])
        for n, r in enumerate(results, 1):
            vals = [_rt.Runtime.value(v) for v in r.values]
            vals = ["; ".join(map(str, (_rt.Runtime.value(x) for x in v))) if isinstance(v, list) else v for v in vals]
            w.writerow([n, *[r.inputs.get(i, "") for i in program.inputs], r.status, len(r.escalated),
                        r.stats.get("tokens", 0), r.run_id, r.problem or "", *vals])
    return results


# ═══════════════════════════════ domains ═════════════════════════════════════
_SYN = {"cv": "resume", "cvs": "resume", "hiring": "resume screening recruit", "recruitment": "resume screening",
        "invoice": "document", "bug": "incident", "outage": "outage incident", "ticket": "helpdesk query",
        "doctor": "clinical", "hospital": "clinical patient", "medicine": "drug prescription",
        "farmer": "farmer crop agriculture", "crops": "crop", "student": "student education",
        "exam": "exam question", "paper": "manuscript research", "journal": "manuscript peer review",
        "loan": "loan credit", "kyc": "kyc customer", "fraud": "fraud", "contract": "contract clause",
        "law": "legal case", "court": "litigation dispute", "phishing": "phishing email", "malware": "alert soc",
        "hack": "alert incident soc", "tender": "tender bids", "citizen": "citizen grievance",
        "complaint": "complaint grievance query", "esg": "esg environmental governance", "carbon": "emissions",
        "shipping": "shipment", "warehouse": "inventory stock", "hotel": "guest booking", "menu": "menu allergen",
        "server": "incident service", "servers": "incident service", "down": "incident",
        "crash": "incident", "website": "incident service", "water": "citizen grievance government",
        "road": "citizen grievance government", "municipal": "citizen grievance government",
        "public": "citizen public government", "electricity": "outage energy meter", "power": "outage energy grid",
        "patient": "patient clinical", "vendor": "supplier", "supplier": "supplier", "salary": "employee hr",
        "employee": "employee hr", "customer": "customer"}


# words that point strongly at one sector; matching packs get a small boost
SECTOR_HINTS = {
    "government": "citizen citizens public municipal water road ration panchayat ministry department government rti",
    "itops": "server servers it network database website app application helpdesk deployment kubernetes",
    "energy": "power electricity grid solar wind meter substation",
    "healthcare": "patient patients hospital clinic doctor nurse clinical",
    "banking": "bank banking loan account card kyc aml",
    "insurance": "insurance claim claims policyholder premium",
    "education": "student students exam course teacher school college university",
    "research": "research thesis manuscript journal grant phd",
    "agriculture": "farmer farmers crop crops soil pest mandi",
    "hr": "employee employees hiring resume cv candidate interview",
    "legal": "contract nda law legal court litigation clause",
    "cyber": "phishing malware vulnerability soc threat breach",
    "retail": "shop store product products ecommerce order returns",
    "hospitality": "hotel guest guests restaurant travel trip booking",
}
_HINTS = {sec: set(words.split()) for sec, words in SECTOR_HINTS.items()}


def _stem(w):
    for suf in ("ations", "ation", "ings", "ing", "ies", "es", "ed", "s"):
        if len(w) > len(suf) + 3 and w.endswith(suf):
            return w[: -len(suf)] + ("y" if suf == "ies" else "")
    return w


def _tokens(text):
    out = []
    for w in re.findall(r"[a-z0-9]+", text.lower()):
        out += [_stem(x) for x in (w + " " + _SYN.get(w, "")).split() if len(x) > 1]
    return out


class _Domains:
    def __init__(self):
        self._cat = None
        self._index = None

    def _load(self):
        if self._cat is None:
            p = HOME / "packs" / "catalog.json"
            self._cat = json.loads(p.read_text(encoding="utf-8")) if p.exists() else {"packs": [], "sectors": {}}
            docs = []
            for m in self._cat["packs"]:
                src = (HOME / m["path"]).read_text(encoding="utf-8") if (HOME / m["path"]).exists() else ""
                goals = " ".join(re.findall(r'goal\s+"([^"]*)"', src))
                text = " ".join([m["title"]] * 3 + [m["description"]] * 2 + [m["sector_name"], m["id"].replace(".", " ").replace("_", " "), goals])
                docs.append(_tokens(text))
            n = len(docs)
            df = {}
            for d in docs:
                for t in set(d):
                    df[t] = df.get(t, 0) + 1
            self._idf = {t: math.log((n + 1) / (c + 0.5)) for t, c in df.items()}
            self._vecs = []
            for d in docs:
                tf = {}
                for t in d:
                    tf[t] = tf.get(t, 0) + 1
                v = {t: (1 + math.log(c)) * self._idf[t] for t, c in tf.items()}
                norm = math.sqrt(sum(x * x for x in v.values())) or 1
                self._vecs.append({t: x / norm for t, x in v.items()})
        return self._cat

    def all(self, sector=None):
        packs = self._load()["packs"]
        return [p for p in packs if sector in (None, p["sector"])]

    def sectors(self):
        cat = self._load()
        counts = {}
        for p in cat["packs"]:
            counts[p["sector"]] = counts.get(p["sector"], 0) + 1
        return [{"id": k, "name": v, "packs": counts.get(k, 0)} for k, v in cat["sectors"].items()]

    def get(self, pack_id):
        for p in self._load()["packs"]:
            if p["id"] == pack_id:
                return p
        raise KeyError(f"no domain pack '{pack_id}' — see al.domains.search(...)")

    def search(self, query, k=5):
        """Best-matching domain packs for a plain-language description (works offline)."""
        packs = self._load()["packs"]
        q = {}
        for t in _tokens(query):
            if t in self._idf:
                q[t] = q.get(t, 0) + self._idf[t]
        norm = math.sqrt(sum(x * x for x in q.values())) or 1
        raw = set(re.findall(r"[a-z0-9]+", query.lower()))
        boost = {sec for sec, hints in _HINTS.items() if raw & hints}
        scored = []
        for m, v in zip(packs, self._vecs):
            s = sum(w / norm * v.get(t, 0) for t, w in q.items())
            if s > 0 and m["sector"] in boost:
                s += 0.15
            if s > 0:
                scored.append((round(s, 3), m))
        scored.sort(key=lambda x: -x[0])
        return [dict(m, score=s) for s, m in scored[:k]]

    def source(self, pack_id):
        return (HOME / self.get(pack_id)["path"]).read_text(encoding="utf-8")


domains = _Domains()


def compose(request: str, k: int = 3):
    """Prompt → plan. Finds the best domain packs for a request and prepares a ready-to-run program.

    Returns dict(best, alternatives, program, inputs) where `program` is a Program and `inputs`
    maps the flow's input to the request text. Works offline; use ask() to customise with AI."""
    matches = domains.search(request, k=k + 1)
    if not matches:
        return {"best": None, "alternatives": [], "program": None, "inputs": {}}
    best = matches[0]
    prog = load(HOME / best["path"])
    return {"best": best, "alternatives": matches[1:], "program": prog,
            "inputs": {prog.inputs[0]: request} if len(prog.inputs) == 1 else {}}


def ask(request: str, current: str | None = None, client=None, log=print):
    """AI writes (or changes) an AlgoLang program; the compiler checks it and errors are repaired.

    Returns dict(source, ok, attempts, diagnostics). Needs ANTHROPIC_API_KEY unless `client` is given."""
    import algoagent
    return algoagent.write_program(request, current=current, client=client, log=log)


# ══════════════════════════════ governance ═══════════════════════════════════
class _Audit:
    verify = staticmethod(_rt.verify_audit)

    @staticmethod
    def report(directory="runs"):
        """Usage, cost and governance summary across every audit log in a folder."""
        d = Path(directory)
        out = {"programs": {}, "agents": {}, "totals": dict(calls=0, tokens=0, escalated=0, blocked=0, failed=0,
                                                            cached=0, tool_calls=0), "chains": {}}
        for f in sorted(d.glob("*.audit.jsonl")):
            out["chains"][f.name] = _rt.verify_audit(f)["ok"]
            for line in f.read_text(encoding="utf-8").splitlines():
                if not line.strip():
                    continue
                r = json.loads(line)
                prog, agent = r.get("program", f.stem), r.get("agent", "?")
                for key, bucket in ((prog, out["programs"]), (f"{prog}/{agent}", out["agents"])):
                    b = bucket.setdefault(key, dict(calls=0, tokens=0, escalated=0, blocked=0, failed=0,
                                                    cached=0, conf_sum=0.0, conf_n=0))
                    st = r.get("status")
                    if st in ("ok", "escalated", "approved"):
                        b["calls"] += 1
                        b["tokens"] += r.get("tokens", 0) or 0
                        if r.get("confidence") is not None:
                            b["conf_sum"] += r["confidence"]
                            b["conf_n"] += 1
                    b["escalated"] += st == "escalated"
                    b["blocked"] += st == "blocked"
                    b["failed"] += st == "failed"
                    b["cached"] += bool(r.get("cached"))
                t = out["totals"]
                st = r.get("status")
                t["calls"] += st in ("ok", "escalated", "approved")
                t["tokens"] += r.get("tokens", 0) or 0
                t["escalated"] += st == "escalated"
                t["blocked"] += st == "blocked"
                t["failed"] += st == "failed"
                t["cached"] += bool(r.get("cached"))
                t["tool_calls"] += len(r.get("tool_calls") or [])
        for bucket in (out["programs"], out["agents"]):
            for b in bucket.values():
                b["avg_confidence"] = round(b.pop("conf_sum") / b["conf_n"], 3) if b["conf_n"] else None
                b.pop("conf_n")
        return out


class _Review:
    @staticmethod
    def pending(path=None):
        return _rt.list_reviews("pending", path)

    @staticmethod
    def all(path=None):
        return _rt.list_reviews("all", path)

    @staticmethod
    def approve(item_id, reviewer="", note="", path=None):
        return _rt.decide_review(item_id, "approved", reviewer, note, path)

    @staticmethod
    def reject(item_id, reviewer="", note="", path=None):
        return _rt.decide_review(item_id, "rejected", reviewer, note, path)


audit = _Audit()
review = _Review()


Overwriting algokit/algolang/__init__.py


**Domain packs** — 120 ready-made multi-agent programs (compressed; collapsed)

In [8]:
import base64, io, tarfile
_PACKS = """
...
MgAB8vEBgOAcZMat/bHEZ/sCAAAAAARZWg==
"""
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(_PACKS)), mode="r:xz") as tf:
    tf.extractall("algokit", filter="data") if hasattr(tarfile, "data_filter") else tf.extractall("algokit")
import glob; print("✓", len(glob.glob("algokit/packs/*/*.algo")), "domain packs unpacked")

✓ 120 domain packs unpacked


In [9]:
import sys
sys.path.insert(0, "algokit")
%load_ext algomagic

In [10]:
%algo_help

```text

algomagic — run AlgoLang inside Jupyter Notebook, JupyterLab and Google Colab.
AlgoProfessor AI R&D Solutions · AlgoLang v0.1

    %load_ext algomagic

    %%algo --arg "topic=Neural networks"      run a whole program in one cell
    %%algo --define                           add agents/models/tools to the session
    %%algo --check                            only check for errors
    %%algo --graph                            also draw the flow as a diagram
    %%algo --python                           also show the generated Python
    %%algo --live                             call real Claude models
    %%algo --interactive                      ask a human to approve low-confidence results

    %algo_run examples/research_brief.algo --arg "topic=..."   run a .algo file
    %algo_list     what is defined in this session
    %algo_reset    forget all --define cells
    %algo_help     this help

After a run, these Python variables are set:
    algo_result   list of values the flow emitted
    algo_calls    list of dicts, one per agent call (agent, model, confidence, status, tokens, output…)

```

## 1. Your first program

One cell holds a whole program: three agents and a `flow main` that wires them together. `par { ... }` runs two agents at the same time.

In [11]:
%%algo --arg "topic=Neural networks"

model smart = "claude-sonnet-5-5"
model fast  = "claude-haiku-5-5"

agent Explainer(topic: text) -> text {
    role  "Friendly professor"
    goal  "Explain {topic} to a first-year student in 5 sentences"
    model fast
}

agent QuizMaker(lesson: text) -> list {
    role  "Exam setter"
    goal  "Write three short quiz questions on this lesson: {lesson}"
    model fast
}

agent Checker(lesson: text) -> score {
    role  "Senior reviewer"
    goal  "Score this lesson 0 to 1 for accuracy: {lesson}"
    model smart
}

flow main(topic: text) {
    lesson = Explainer(topic)
    [quiz, quality] = par { QuizMaker(lesson), Checker(lesson) }
    emit lesson, quiz, quality
}

▶ flow main  run=f62a44fc  backend=mock


    topic = 'Neural networks'


  → Explainer          conf=0.88  «Explainer #4348» Explain Neural networks to a first-year student in 5 s


  → QuizMaker          conf=0.78  [3 items]


  → Checker            conf=0.89  0.61


Agent,Model,Conf.,Status,Tokens,Output
Explainer,claude-haiku-5-5,0.88,ok,86,«Explainer #4348» Explain Neural networks to a first-year student in 5 sentences
QuizMaker,claude-haiku-5-5,0.78,ok,101,item 1 for: Write three short quiz questions on this lesson: «Explainer #4348» Exp…item 2 for: Write three short quiz questions on this lesson: «Explainer #4348» Exp…item 3 for: Write three short quiz questions on this lesson: «Explainer #4348» Exp…
Checker,claude-sonnet-5-5,0.89,ok,99,0.61


## 2. Build agents cell by cell

Notebook style: define models, tools and agents in separate `--define` cells, then run a short `flow main` that uses them. Re-running a define cell replaces the old definition. `%algo_list` shows what the session holds; `%algo_reset` clears it.

In [12]:
%%algo --define
model smart = "claude-sonnet-5-5"
model fast  = "claude-haiku-5-5"
tool  web.search

In [13]:
%%algo --define
agent Researcher(topic: text) -> text {
    role  "Senior research analyst"
    goal  "Find five verified, recent facts about {topic}, with sources"
    uses  web.search
    model smart
    policy {
        budget tokens 6000
        retry 2
    }
}

agent Writer(facts: text, tone: text = "academic") -> text {
    role  "Technical writer"
    goal  "Write a 200-word brief from these facts in a {tone} tone: {facts}"
    model fast
}

In [14]:
%%algo --define
agent Reviewer(draft: text) -> score {
    role  "Peer reviewer"
    goal  "Score this draft from 0 to 1 for factual accuracy and clarity: {draft}"
    model smart
    policy {
        require human when confidence < 0.70
    }
}

In [15]:
%algo_list

#,Defines,Size
1,(models / tools),4 lines
2,"Researcher, Writer",17 lines
3,Reviewer,9 lines


Now the flow. `--graph` draws how the agents connect before running them. The Reviewer's score decides whether the Writer redrafts.

In [16]:
%%algo --arg "topic=Blockchain data integrity" --graph

flow main(topic: text) {
    facts = Researcher(topic)
    [formal, simple] = par {
        Writer(facts),
        Writer(facts, tone = "plain-English")
    }
    quality = Reviewer(formal)
    if quality < 0.80 {
        formal = Writer(facts, tone = "rigorous academic")
    }
    emit formal, simple
}

▶ flow main  run=5b3ae2aa  backend=mock


    topic = 'Blockchain data integrity'


  → Researcher         conf=0.83  «Researcher #5704» Find five verified, recent facts about Blockchain dat


  → Writer             conf=0.71  «Writer #7140» Write a 200-word brief from these facts in a academic ton


  → Writer             conf=0.77  «Writer #6561» Write a 200-word brief from these facts in a plain-Englis


  → Reviewer           conf=0.68  0.78  ⚠ ESCALATED to human (conf < 0.70)


  → Writer             conf=0.84  «Writer #7619» Write a 200-word brief from these facts in a rigorous aca


Agent,Model,Conf.,Status,Tokens,Output
Researcher,claude-sonnet-5-5,0.83,ok,93,"«Researcher #5704» Find five verified, recent facts about Blockchain data integrity, with…"
Writer,claude-haiku-5-5,0.71,ok,107,«Writer #7140» Write a 200-word brief from these facts in a academic tone: «Researche…
Writer,claude-haiku-5-5,0.77,ok,109,«Writer #6561» Write a 200-word brief from these facts in a plain-English tone: «Rese…
Reviewer,claude-sonnet-5-5,0.68,⚠ human review,106,0.78
Writer,claude-haiku-5-5,0.84,ok,110,«Writer #7619» Write a 200-word brief from these facts in a rigorous academic tone: «…


## 3. Use the results in Python

After every run, `algo_result` holds what the flow emitted and `algo_calls` holds one record per agent call.

In [17]:
print("First emitted value:\n", algo_result[0])
try:
    import pandas as pd
    display(pd.DataFrame(algo_calls)[["agent", "model", "confidence", "status", "tokens"]])
except ImportError:
    for c in algo_calls:
        print(c["agent"], c.get("confidence"), c["status"])

First emitted value:
 «Writer #7619» Write a 200-word brief from these facts in a rigorous academic tone: «…


,agent,model,confidence,status,tokens
0,Researcher,claude-sonnet-5-5,0.83,ok,93
1,Writer,claude-haiku-5-5,0.71,ok,107
2,Writer,claude-haiku-5-5,0.77,ok,109
3,Reviewer,claude-sonnet-5-5,0.68,escalated,106
4,Writer,claude-haiku-5-5,0.84,ok,110


## 4. A multi-agent team: paper review

A Planner splits the job into criteria; `par for` starts one reviewer agent per criterion, all in parallel. The Editor's policy **forbids** decisions that see author identity.

In [18]:
%%algo --define
tool file.read

agent Planner(paper: text) -> list {
    role  "Associate editor"
    goal  "List the review criteria that matter most for this manuscript: {paper}"
    uses  file.read
    model smart
}

agent CriterionReviewer(criterion: text, paper: text) -> text {
    role  "Domain reviewer"
    goal  "Review {paper} strictly on: {criterion}. Give strengths, weaknesses, fixes."
    uses  file.read, web.search
    model fast
}

agent EthicsAuditor(paper: text) -> text {
    role  "Responsible-AI auditor"
    goal  "Check {paper} for dataset consent, bias, reproducibility and dual-use risk"
    model smart
}

agent Editor(reports: list, ethics: text) -> score {
    role  "Editor-in-Chief"
    goal  "Weigh {reports} and {ethics}; score acceptance likelihood 0 to 1"
    model smart
    policy {
        require human when confidence < 0.75
        forbid gender, caste, religion, nationality
    }
}

agent DecisionLetter(reports: list, verdict: score) -> text {
    role  "Editorial assistant"
    goal  "Draft a courteous decision letter (score {verdict}) summarising {reports}"
    model fast
}

In [19]:
%%algo --arg "paper=LG-CEF: lesion-guided ensemble for diabetic retinopathy grading on IDRiD" --graph

flow main(paper: text) {
    criteria = Planner(paper)
    reports  = par for c in criteria { CriterionReviewer(c, paper) }
    ethics   = EthicsAuditor(paper)
    verdict  = Editor(reports, ethics)
    letter   = DecisionLetter(reports, verdict)
    emit verdict, letter
}

▶ flow main  run=8ba41810  backend=mock


    paper = 'LG-CEF: lesion-guided ensemble for diabetic retinopathy grading on IDRiD'


  → Planner            conf=0.93  [3 items]


  → CriterionReviewer  conf=0.69  «CriterionReviewer #7954» Review LG-CEF: lesion-guided ensemble for diab


  → CriterionReviewer  conf=0.86  «CriterionReviewer #6318» Review LG-CEF: lesion-guided ensemble for diab


  → CriterionReviewer  conf=0.76  «CriterionReviewer #7148» Review LG-CEF: lesion-guided ensemble for diab


  → EthicsAuditor      conf=0.63  «EthicsAuditor #0740» Check LG-CEF: lesion-guided ensemble for diabetic 


  → Editor             conf=0.72  0.67  ⚠ ESCALATED to human (conf < 0.75)


  → DecisionLetter     conf=0.90  «DecisionLetter #3023» Draft a courteous decision letter (score 0.67) su


Agent,Model,Conf.,Status,Tokens,Output
Planner,claude-sonnet-5-5,0.93,ok,105,Novelty & contributionMethodology & rigourExperimental validity
CriterionReviewer,claude-haiku-5-5,0.69,ok,112,«CriterionReviewer #7954» Review LG-CEF: lesion-guided ensemble for diabetic retinopathy grading… [file.read: no file name found in the request]
CriterionReviewer,claude-haiku-5-5,0.86,ok,111,«CriterionReviewer #6318» Review LG-CEF: lesion-guided ensemble for diabetic retinopathy grading… [file.read: no file name found in the request]
CriterionReviewer,claude-haiku-5-5,0.76,ok,111,«CriterionReviewer #7148» Review LG-CEF: lesion-guided ensemble for diabetic retinopathy grading… [file.read: no file name found in the request]
EthicsAuditor,claude-sonnet-5-5,0.63,ok,106,«EthicsAuditor #0740» Check LG-CEF: lesion-guided ensemble for diabetic retinopathy grading …
Editor,claude-sonnet-5-5,0.72,⚠ human review,214,0.67
DecisionLetter,claude-haiku-5-5,0.90,ok,195,«DecisionLetter #3023» Draft a courteous decision letter (score 0.67) summarising «CriterionR…


## 5. The ethics guard in action

Same team, different manuscript. Its title mentions *nationality*, which reaches the Editor through the reviewers' reports. The runtime stops the run before the Editor decides, and records the block in the audit log.

In [20]:
%%algo --arg "paper=Does author nationality predict citation counts?"

flow main(paper: text) {
    criteria = Planner(paper)
    reports  = par for c in criteria { CriterionReviewer(c, paper) }
    ethics   = EthicsAuditor(paper)
    verdict  = Editor(reports, ethics)
    emit verdict
}

▶ flow main  run=b28bfb3c  backend=mock


    paper = 'Does author nationality predict citation counts?'


  → Planner            conf=0.71  [4 items]


  → CriterionReviewer  conf=0.70  «CriterionReviewer #1393» Review Does author nationality predict citatio


  → CriterionReviewer  conf=0.80  «CriterionReviewer #1033» Review Does author nationality predict citatio


  → CriterionReviewer  conf=0.75  «CriterionReviewer #9170» Review Does author nationality predict citatio


  → CriterionReviewer  conf=0.81  «CriterionReviewer #4612» Review Does author nationality predict citatio


  → EthicsAuditor      conf=0.79  «EthicsAuditor #1139» Check Does author nationality predict citation cou


Agent,Model,Conf.,Status,Tokens,Output
Planner,claude-sonnet-5-5,0.71,ok,99,Novelty & contributionMethodology & rigourExperimental validityRelated-work coverage
CriterionReviewer,claude-haiku-5-5,0.70,ok,106,«CriterionReviewer #1393» Review Does author nationality predict citation counts? strictly on: N… [file.read: no file name found in the request]
CriterionReviewer,claude-haiku-5-5,0.80,ok,105,«CriterionReviewer #1033» Review Does author nationality predict citation counts? strictly on: M… [file.read: no file name found in the request]
CriterionReviewer,claude-haiku-5-5,0.81,ok,105,«CriterionReviewer #4612» Review Does author nationality predict citation counts? strictly on: R… [file.read: no file name found in the request]
CriterionReviewer,claude-haiku-5-5,0.75,ok,105,«CriterionReviewer #9170» Review Does author nationality predict citation counts? strictly on: E… [file.read: no file name found in the request]
EthicsAuditor,claude-sonnet-5-5,0.79,ok,100,«EthicsAuditor #1139» Check Does author nationality predict citation counts? for dataset con…
Editor,—,—,blocked,—,forbidden attribute 'nationality'


## 6. The compiler catches mistakes

Nothing runs until the program is correct. This cell has six deliberate errors; each is reported with its line in this cell.

In [21]:
%%algo
agent Summariser(doc: text) -> text {
    role "Summariser"
    goal "Summarise {document}"
    uses email.send
}

agent Screener(caste: text) -> score {
    goal "Score applicant {caste}"
    policy { forbid caste }
}

flow main(doc: text) {
    s = Summariser(doc, doc)
    t = summarise_text(doc)
    if s < 0.5 { emit s }
}

Code,Where,Problem
E109,line 1,agent Summariser: goal mentions {document} but there is no parameter 'document'
E104,line 1,"agent Summariser uses tool 'email.send', which is not declared (add `tool email.send`)"
E108,line 7,agent Screener takes forbidden attribute 'caste' as an input
E102,line 13,"Summariser takes 1 argument(s) (doc), got 2 positional"
E101,line 14,"'summarise_text' is not an agent — in AlgoLang every unit of work is an agent (declare it with `agent summarise_text(...) -> type { goal ""..."" }`)"
E110,line 15,"'s' is text, not a score — only scores/numbers can be compared"


## 7. Human in the loop

With `--interactive`, a result whose confidence is under the agent's threshold pauses the run and asks you to approve it. Type `y` and press Enter to approve; anything else keeps it flagged for review.

In [22]:
%%algo --arg "topic=Blockchain data integrity" --interactive

flow main(topic: text) {
    facts   = Researcher(topic)
    draft   = Writer(facts)
    quality = Reviewer(draft)
    emit draft, quality
}

▶ flow main  run=8111e682  backend=mock


    topic = 'Blockchain data integrity'


  → Researcher         conf=0.83  «Researcher #5704» Find five verified, recent facts about Blockchain dat


  → Writer             conf=0.71  «Writer #7140» Write a 200-word brief from these facts in a academic ton


  → Reviewer           conf=0.68  0.78  ⚠ ESCALATED to human (conf < 0.70)


Agent,Model,Conf.,Status,Tokens,Output
Researcher,claude-sonnet-5-5,0.83,ok,93,"«Researcher #5704» Find five verified, recent facts about Blockchain data integrity, with…"
Writer,claude-haiku-5-5,0.71,ok,107,«Writer #7140» Write a 200-word brief from these facts in a academic tone: «Researche…
Reviewer,claude-sonnet-5-5,0.68,⚠ human review,106,0.78


## 8. Check only, or see the generated Python

`--check` compiles without running. `--python` shows the Python that AlgoLang generates: each agent becomes a governed object and each flow a function.

In [23]:
%%algo --python --arg "topic=Graph machine learning"

flow main(topic: text) {
    facts = Researcher(topic)
    emit Writer(facts, tone = "simple")
}

**Generated Python**

```python
# Generated by algoc v0.2 from notebook.algo — do not edit; edit the .algo source.
import os, sys
sys.path.insert(0, os.environ.get('ALGO_HOME') or 'algokit')
from algoruntime import Runtime, AgentSpec

rt = Runtime(program='notebook', **globals().get('__algo_runtime__', {}))
rt.model('smart', 'claude-sonnet-5-5')
rt.model('fast', 'claude-haiku-5-5')
rt.tool('file.read')
rt.tool('web.search')

Researcher = rt.agent(AgentSpec(
    name='Researcher', params=[('topic', 'text', None)], returns='text',
    role='Senior research analyst',
    goal='Find five verified, recent facts about {topic}, with sources',
    uses=['web.search'], model='smart', policy={'budget_tokens': 6000, 'retry': 2}))

Writer = rt.agent(AgentSpec(
    name='Writer', params=[('facts', 'text', None), ('tone', 'text', 'academic')], returns='text',
    role='Technical writer',
    goal='Write a 200-word brief from these facts in a {tone} tone: {facts}',
    uses=[], model='fast', policy={}))

Reviewer = rt.agent(AgentSpec(
    name='Reviewer', params=[('draft', 'text', None)], returns='score',
    role='Peer reviewer',
    goal='Score this draft from 0 to 1 for factual accuracy and clarity: {draft}',
    uses=[], model='smart', policy={'human_below': 0.7}))

Planner = rt.agent(AgentSpec(
    name='Planner', params=[('paper', 'text', None)], returns='list',
    role='Associate editor',
    goal='List the review criteria that matter most for this manuscript: {paper}',
    uses=['file.read'], model='smart', policy={}))

CriterionReviewer = rt.agent(AgentSpec(
    name='CriterionReviewer', params=[('criterion', 'text', None), ('paper', 'text', None)], returns='text',
    role='Domain reviewer',
    goal='Review {paper} strictly on: {criterion}. Give strengths, weaknesses, fixes.',
    uses=['file.read', 'web.search'], model='fast', policy={}))

EthicsAuditor = rt.agent(AgentSpec(
    name='EthicsAuditor', params=[('paper', 'text', None)], returns='text',
    role='Responsible-AI auditor',
    goal='Check {paper} for dataset consent, bias, reproducibility and dual-use risk',
    uses=[], model='smart', policy={}))

Editor = rt.agent(AgentSpec(
    name='Editor', params=[('reports', 'list', None), ('ethics', 'text', None)], returns='score',
    role='Editor-in-Chief',
    goal='Weigh {reports} and {ethics}; score acceptance likelihood 0 to 1',
    uses=[], model='smart', policy={'human_below': 0.75, 'forbid': ['gender', 'caste', 'religion', 'nationality']}))

DecisionLetter = rt.agent(AgentSpec(
    name='DecisionLetter', params=[('reports', 'list', None), ('verdict', 'score', None)], returns='text',
    role='Editorial assistant',
    goal='Draft a courteous decision letter (score {verdict}) summarising {reports}',
    uses=[], model='fast', policy={}))

def flow_main(topic):
    rt.begin_flow('main', {'topic': topic})
    facts = Researcher(topic=topic)
    rt.emit(Writer(facts=facts, tone='simple'))
    return rt.end_flow()

if __name__ == "__main__":
    rt.cli(flow_main, ['topic'])

```

▶ flow main  run=732bbfe0  backend=mock


    topic = 'Graph machine learning'


  → Researcher         conf=0.85  «Researcher #9894» Find five verified, recent facts about Graph machine 


  → Writer             conf=0.79  «Writer #1660» Write a 200-word brief from these facts in a simple tone:


Agent,Model,Conf.,Status,Tokens,Output
Researcher,claude-sonnet-5-5,0.85,ok,92,"«Researcher #9894» Find five verified, recent facts about Graph machine learning, with so…"
Writer,claude-haiku-5-5,0.79,ok,107,«Writer #1660» Write a 200-word brief from these facts in a simple tone: «Researcher …


## 9. Real Claude models (optional)

1. Get an API key at [console.anthropic.com](https://console.anthropic.com).
2. **Colab:** click the 🔑 *Secrets* icon in the left sidebar, add a secret named `ANTHROPIC_API_KEY`, and allow this notebook to use it. **Jupyter:** set `ANTHROPIC_API_KEY` in the terminal before starting Jupyter.
3. Run the next two cells. Live calls use API credits; keep `budget tokens` policies on your agents.

In [24]:
%pip install -q anthropic
import os
try:
    from google.colab import userdata          # Colab secrets
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("✓ key loaded from Colab secrets")
except Exception:
    print("Using ANTHROPIC_API_KEY from the environment" if os.environ.get("ANTHROPIC_API_KEY")
          else "No API key found — the live cell below will say so and stop safely.")

Note: you may need to restart the kernel to use updated packages.
No API key found — the live cell below will say so and stop safely.


In [25]:
%%algo --live --arg "topic=Agentic AI in Indian higher education"

flow main(topic: text) {
    facts = Researcher(topic)
    brief = Writer(facts)
    emit brief
}

## 10. Enterprise: the Python SDK and 120 domain packs

Version 0.2 adds a Python SDK, `import algolang`. It is the same engine that drives all three modes: the `algo` command line, the graphical Studio and the Prompt UI.

Start by searching the domain library in plain English.

In [26]:
import algolang as al
print("AlgoLang", al.__version__, "·", len(al.domains.all()), "domains in", len(al.domains.sectors()), "sectors")
for d in al.domains.search("our servers are down and customers cannot log in")[:3]:
    print(f"  {d['score']:.2f}  {d['id']:<32} {d['title']}")

AlgoLang 0.2.0 · 120 domains in 20 sectors
  0.44  itops.incident_triage            IT incident triage
  0.35  itops.log_anomaly                Log anomaly detection
  0.33  itops.postmortem                 Incident postmortem writer


Load a pack, look at its agents, and run it. Every run returns values, per-agent calls, statistics and an audit trail.

In [27]:
p = al.pack("insurance.claims_triage")
print("inputs:", p.inputs, "· agents:", list(p.agents))
r = p.run(quiet=True, claim="Rear-end collision in Pune, bumper damage, no injuries")
print("status:", r.status, "· calls escalated to a human:", len(r.escalated))
for c in r.calls:
    print(f"  {c['agent']:<14} conf={c['confidence']:.2f}")

inputs: ['claim'] · agents: ['Complexity', 'Coverage', 'Intake', 'Routing']
status: ok · calls escalated to a human: 1
  Intake         conf=0.89
  Coverage       conf=0.80
  Complexity     conf=0.75
  Routing        conf=0.67


### Batch processing at scale
`run_batch` runs the program over many records at once, spread across worker threads. Org-wide limits still apply to every record, and each one is written to the hash-chained audit log.

In [28]:
rows = [{"claim": f"Claim {i}: water damage after pipe burst, flat {i}"} for i in range(50)]
import time; t = time.time()
results = p.run_batch(rows, workers=8, quiet=True)
print(len(results), "claims in", round(time.time() - t, 2), "s ·",
      sum(bool(x.escalated) for x in results), "sent for human review")

50 claims in 0.16 s · 39 sent for human review


### Governance: org policy, audit chain, review queue
`configure` sets rules for the whole organisation, and they merge with every program's own policy (the stricter rule wins). The audit log is SHA-256 hash-chained, so any edit to it is detected.

In [29]:
al.configure(forbid=["salary", "caste"], min_human_below=0.75, max_budget_tokens=8000)
print("audit chain:", al.audit.verify(r.audit_path))
print("pending reviews:", len(al.review.pending()))
for item in al.review.pending()[:2]:
    al.review.approve(item["id"], reviewer="Dr. S. Satyanarayana", note="checked")
print("after approving two:", len(al.review.pending()))

audit chain: {'ok': True, 'records': 408, 'first_bad_line': None, 'reason': ''}
pending reviews: 85
after approving two: 83


### Your own tools
Register a Python function as a tool (it receives the agent's request text). Agents can use it only if they declare it with `uses`.

In [30]:
@al.tool("company.directory", "Look up an employee's team")
def directory(query):
    teams = {"Asha": "Data Science", "Ravi": "IT Operations"}
    return next((f"{n} → {t}" for n, t in teams.items() if n in query), "not in directory")

prog = al.compile('''
tool company.directory
agent Router(name: text) -> text {
    role "HR helpdesk router"
    goal "Route a request from {name} to the right team"
    uses company.directory
}
flow main(name: text) { t = Router(name)  emit t }
''')
print(prog.run(quiet=True, name="Ravi").values[0])

«Router #1685» Route a request from Ravi to the right team  [company.directory: Ravi → IT Operations]


### Prompt mode from Python
`compose` turns a plain-English request into a governed program: it picks the best pack and returns the source so you can edit it.

In [31]:
plan = al.compose("screen this CV fairly for a junior data analyst role")
print("best match:", plan["best"]["id"], "· alternatives:", [a["id"] for a in plan["alternatives"]])
out = plan["program"].run(quiet=True, **plan["inputs"])
print("status:", out.status, "· agents run:", [c["agent"] for c in out.calls])

best match: hr.resume_screening · alternatives: ['hr.job_description', 'hr.interview_kit', 'hr.employee_router']
status: ok · agents run: ['SkillsReviewer', 'ExperienceReviewer', 'PotentialReviewer', 'HiringPanel', 'Note']


## 11. Exercises

1. **New agent.** Add a `Translator(text: text, language: text) -> text` agent and translate the brief from section 2 into Hindi and Marathi in a `par` block.
2. **Policy.** Give `Writer` a `budget tokens 50` policy and run section 2 again. What happens, and why is it better than discovering the cost afterwards?
3. **Enterprise.** Pick any pack with `al.domains.search(...)`, run it over a 100-row batch, then open the Studio's Governance tab (`algo studio`) to approve the escalated items.
4. **Fairness.** Design a scholarship-screening flow with an agent that must never see `gender` or `caste`. Try to break it — the compiler (E108) and the runtime should both stop you.

---
<div style="color:#0b1f3a;font-size:12px">AlgoLang v0.2 Enterprise · AlgoProfessor AI R&amp;D Solutions, Hyderabad · <span style="color:#c9a227">www.algoprofessor.com</span></div>